<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 03 — Book Inventory

### شناسایی محیط پروژه و فایل‌های فیزیکی کتاب

هدف این Notebook ساخت یک Inventory قابل‌اعتماد برای منابع کتابی پروژه است؛ پیش از آنکه در Notebook بعدی وارد مرحله `Book Ingestion` شویم.

در این مرحله اول فقط وضعیت واقعی فایل‌سیستم بررسی می‌شود:

- ریشه پروژه بدون استفاده از مسیر Hard-code شناسایی می‌شود.
- پوشه `data/books/` بررسی می‌شود.
- تمام فایل‌های فیزیکی `DOCX` موجود شناسایی می‌شوند.
- تعداد، نام و حجم هر فایل ثبت می‌شود.

در این مرحله هنوز درباره تعداد آثار مستقل، نسخه‌ها، نویسنده، حقوق، یا فایل Canonical هیچ فرضی نمی‌کنیم.

اصل مهم این Notebook:

`Work != Edition != Physical File`

بنابراین تعداد فایل‌های پیدا‌شده الزاماً برابر با تعداد کتاب‌ها یا آثار مستقل نیست.

این Cell فقط `Discovery` انجام می‌دهد و هیچ فایل یا داده‌ای را تغییر نمی‌دهد.

**معیار PASS:**

- ریشه پروژه از روی ساختار واقعی Repository پیدا شود.
- پوشه `data/books/` موجود باشد.
- حداقل یک فایل `DOCX` واقعی شناسایی شود.

پس از اجرای Cell، Output واقعی بررسی می‌شود و فقط بعد از آن وارد مرحله بعد خواهیم شد.

</div>

In [1]:
# Code Cell 01

from pathlib import Path

current_path = Path.cwd().resolve()
search_paths = [current_path, *current_path.parents]

project_root = next(
    (
        path
        for path in search_paths
        if (path / "data" / "books").is_dir()
        and (path / "notebooks" / "phase2").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be detected from the current working directory."
    )

books_dir = project_root / "data" / "books"
book_files = sorted(books_dir.glob("*.docx"), key=lambda path: path.name.casefold())

print("Current working directory:", current_path)
print("Project root:", project_root)
print("Books directory:", books_dir)
print("DOCX files found:", len(book_files))
print()

for index, path in enumerate(book_files, start=1):
    size_kb = path.stat().st_size / 1024
    print(f"{index:02d}. {path.name}")
    print(f"    Size: {size_kb:,.1f} KB")

print()
print("BOOK FILE DISCOVERY GATE:", "PASS" if book_files else "FAIL")

Current working directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2
Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Books directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/books
DOCX files found: 7

01. Money and ME By Komeil Roudi 040214.docx
    Size: 170.0 KB
02. Money_and_Me_FA_Published_Final_RTL_FIXED.docx
    Size: 168.7 KB
03. درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
    Size: 350.1 KB
04. سواد مالی برای همه.docx
    Size: 179.9 KB
05. سواد_مالی_برای_همه_نسخه_نهایی_بازسازی_کامل.docx
    Size: 170.4 KB
06. نوشتار آموزشی تربیت مربی سواد مالی.docx
    Size: 121.0 KB
07. نوشتار_آموزشی_تربیت_مربی_سواد_مالی_نسخه_ورد_بازطراحی_RTL.docx
    Size: 120.1 KB

BOOK FILE DISCOVERY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### مقایسه سریع محتوایی و شناسایی نسخه‌های احتمالی یک اثر

استخراج اولیه ۷ فایل با موفقیت انجام شد، اما مقایسه مستقیم Character-by-Character با `SequenceMatcher` برای متن‌های بلند کارایی مناسبی نداشت و متوقف شد.

این یک مشکل Performance در Validator بود و نشانه خرابی فایل‌ها نیست.

از خروجی قبلی یک نتیجه قطعی داریم:

- فایل‌های شماره ۶ و ۷ دارای `Text SHA-256` کاملاً یکسان هستند؛ بنابراین متن استخراج‌شده آن‌ها دقیقاً یکسان است، هرچند فایل فیزیکی آن‌ها متفاوت است.
- فایل‌های شماره ۴ و ۵ نیز از نظر تعداد Paragraph و حجم متن بسیار نزدیک هستند، اما رابطه آن‌ها هنوز قطعی اعلام نمی‌شود.

در این مرحله برای مقایسه سریع‌تر از `5-word shingles` استفاده می‌کنیم.

برای هر جفت فایل دو معیار محاسبه می‌شود:

- `Jaccard`: میزان شباهت کلی محتوای دو فایل
- `Containment`: میزان قرار گرفتن محتوای فایل کوچک‌تر در فایل بزرگ‌تر

این معیارها برای Discovery و پیدا کردن Candidateهای مربوط به یک Work استفاده می‌شوند و هنوز به‌تنهایی تعیین‌کننده `Work ID` یا نسخه Canonical نیستند.

هیچ فایل یا Artifactی در این مرحله تغییر نمی‌کند.

**معیار PASS:**

تمام ۲۱ جفت ممکن بین ۷ فایل مقایسه شوند و نتیجه بدون کندی غیرعادی تولید شود.

</div>

In [2]:
# Code Cell 02

import hashlib
import re
import unicodedata
import zipfile
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"

def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_docx_text(path):
    with zipfile.ZipFile(path) as archive:
        root = ET.fromstring(archive.read("word/document.xml"))

    paragraphs = []

    for paragraph in root.findall(f".//{{{W_NS}}}p"):
        text = "".join(
            node.text or ""
            for node in paragraph.findall(f".//{{{W_NS}}}t")
        ).strip()

        if text:
            paragraphs.append(text)

    full_text = "\n".join(paragraphs)
    normalized = normalize_text(full_text)

    return {
        "path": path,
        "paragraph_count": len(paragraphs),
        "text_chars": len(full_text),
        "text": normalized,
        "text_sha256": hashlib.sha256(
            normalized.encode("utf-8")
        ).hexdigest(),
    }

def make_shingles(text, size=5):
    tokens = re.findall(r"\w+", text, flags=re.UNICODE)

    if len(tokens) < size:
        return {tuple(tokens)} if tokens else set()

    return {
        tuple(tokens[i:i + size])
        for i in range(len(tokens) - size + 1)
    }

book_inventory = [
    extract_docx_text(path)
    for path in book_files
]

for item in book_inventory:
    item["shingles"] = make_shingles(item["text"])

comparisons = []

for i in range(len(book_inventory)):
    for j in range(i + 1, len(book_inventory)):
        left = book_inventory[i]
        right = book_inventory[j]

        exact_text = left["text_sha256"] == right["text_sha256"]

        intersection = len(left["shingles"] & right["shingles"])
        union = len(left["shingles"] | right["shingles"])
        smaller = min(len(left["shingles"]), len(right["shingles"]))

        jaccard = intersection / union if union else 1.0
        containment = intersection / smaller if smaller else 1.0

        if exact_text:
            status = "EXACT TEXT DUPLICATE"
        elif jaccard >= 0.90 and containment >= 0.95:
            status = "NEAR-DUPLICATE CANDIDATE"
        elif jaccard >= 0.65 and containment >= 0.80:
            status = "SAME-WORK CANDIDATE"
        else:
            status = "UNRESOLVED"

        comparisons.append({
            "left": i + 1,
            "right": j + 1,
            "jaccard": jaccard,
            "containment": containment,
            "status": status,
        })

comparisons = sorted(
    comparisons,
    key=lambda x: (x["containment"], x["jaccard"]),
    reverse=True,
)

print("FAST PAIRWISE CONTENT COMPARISON")
print("-" * 88)

for item in comparisons:
    print(
        f"[{item['left']}] ↔ [{item['right']}]  "
        f"Jaccard: {item['jaccard']:.4f}  "
        f"Containment: {item['containment']:.4f}  "
        f"| {item['status']}"
    )

print()
print("Pairs evaluated:", len(comparisons))
print(
    "FAST CONTENT COMPARISON GATE:",
    "PASS" if len(comparisons) == 21 else "FAIL",
)

FAST PAIRWISE CONTENT COMPARISON
----------------------------------------------------------------------------------------
[6] ↔ [7]  Jaccard: 1.0000  Containment: 1.0000  | EXACT TEXT DUPLICATE
[4] ↔ [5]  Jaccard: 0.9953  Containment: 0.9982  | NEAR-DUPLICATE CANDIDATE
[3] ↔ [5]  Jaccard: 0.0133  Containment: 0.0373  | UNRESOLVED
[3] ↔ [4]  Jaccard: 0.0133  Containment: 0.0372  | UNRESOLVED
[2] ↔ [5]  Jaccard: 0.0019  Containment: 0.0045  | UNRESOLVED
[2] ↔ [4]  Jaccard: 0.0019  Containment: 0.0045  | UNRESOLVED
[2] ↔ [3]  Jaccard: 0.0007  Containment: 0.0016  | UNRESOLVED
[3] ↔ [6]  Jaccard: 0.0005  Containment: 0.0013  | UNRESOLVED
[3] ↔ [7]  Jaccard: 0.0005  Containment: 0.0013  | UNRESOLVED
[2] ↔ [6]  Jaccard: 0.0002  Containment: 0.0005  | UNRESOLVED
[2] ↔ [7]  Jaccard: 0.0002  Containment: 0.0005  | UNRESOLVED
[5] ↔ [6]  Jaccard: 0.0002  Containment: 0.0004  | UNRESOLVED
[5] ↔ [7]  Jaccard: 0.0002  Containment: 0.0004  | UNRESOLVED
[4] ↔ [6]  Jaccard: 0.0002  Containment: 0.0004 

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ثبت روابط شناخته‌شده بین Work، نسخه اصلی و نسخه‌های بازسازی‌شده

پس از مقایسه محتوایی فایل‌ها، اطلاعات تکمیلی درباره منشأ آن‌ها مشخص شد.

بر اساس اطلاعات انسانی تأییدشده:

- فایل `[6]` نسخه اصلی دریافت‌شده از مولف است و فایل `[7]` نسخه بازطراحی‌شده همان محتواست.
- فایل `[4]` نسخه اصلی دریافت‌شده از مولف است و فایل `[5]` نسخه بازسازی‌شده همان اثر است.
- فایل `[1]` نسخه انگلیسی اصلی دریافت‌شده از مولف است.
- فایل `[2]` ترجمه فارسی ساخته‌شده از فایل `[1]` است و نسخه اصلی مولف محسوب نمی‌شود.

بنابراین فایل‌های بازسازی‌شده یا ترجمه‌شده نباید با نسخه‌های اصلی مولف اشتباه گرفته شوند.

در این مرحله هیچ فایلی حذف نمی‌شود. هدف فقط ثبت وضعیت Inventory و حفظ Lineage است.

برای پروژه فارسی، فایل `[2]` می‌تواند در آینده Candidate مناسب برای Ingestion باشد، اما پذیرش آن نیازمند Validation مستقل کیفیت ترجمه است.

این مرحله همچنین بررسی می‌کند که فایل‌هایی که در روابط بالا ذکر شده‌اند واقعاً در Inventory موجود باشند و Fingerprintهای قبلی با این روابط سازگار باشند.

**معیار PASS:**

- تمام روابط اعلام‌شده به فایل‌های واقعی موجود اشاره کنند.
- `[6]` و `[7]` همچنان Exact Text Duplicate باشند.
- `[4]` و `[5]` همچنان Near-Duplicate باشند.
- نقش نسخه اصلی، نسخه بازسازی‌شده و ترجمه بدون ابهام ثبت شود.

</div>

In [3]:
# Code Cell 03

known_relationships = [
    {
        "work_group": "money_and_me",
        "file_index": 1,
        "role": "AUTHOR_SOURCE_ENGLISH",
        "canonical_status": "SOURCE",
    },
    {
        "work_group": "money_and_me",
        "file_index": 2,
        "role": "USER_CREATED_PERSIAN_TRANSLATION",
        "canonical_status": "TRANSLATION_QA_REQUIRED",
    },
    {
        "work_group": "financial_literacy_for_all",
        "file_index": 4,
        "role": "AUTHOR_SOURCE",
        "canonical_status": "SOURCE",
    },
    {
        "work_group": "financial_literacy_for_all",
        "file_index": 5,
        "role": "USER_REVISED_VERSION",
        "canonical_status": "NON_CANONICAL",
    },
    {
        "work_group": "trainer_training_text",
        "file_index": 6,
        "role": "AUTHOR_SOURCE",
        "canonical_status": "SOURCE",
    },
    {
        "work_group": "trainer_training_text",
        "file_index": 7,
        "role": "USER_REVISED_VERSION",
        "canonical_status": "NON_CANONICAL",
    },
]

comparison_lookup = {
    (min(item["left"], item["right"]), max(item["left"], item["right"])): item
    for item in comparisons
}

checks = []

for relation in known_relationships:
    file_index = relation["file_index"]
    exists = 1 <= file_index <= len(book_files)

    checks.append(exists)

    print(
        f"[{file_index}] {book_files[file_index - 1].name if exists else 'MISSING'}"
    )
    print("    Work group:", relation["work_group"])
    print("    Role:", relation["role"])
    print("    Canonical status:", relation["canonical_status"])
    print()

pair_6_7 = comparison_lookup[(6, 7)]
pair_4_5 = comparison_lookup[(4, 5)]

duplicate_6_7_ok = pair_6_7["status"] == "EXACT TEXT DUPLICATE"
duplicate_4_5_ok = pair_4_5["status"] == "NEAR-DUPLICATE CANDIDATE"

print("RELATIONSHIP VALIDATION")
print("-" * 80)
print("[6] ↔ [7]:", pair_6_7["status"])
print("[4] ↔ [5]:", pair_4_5["status"])
print()
print("All referenced files exist:", all(checks))
print("6/7 exact duplicate confirmed:", duplicate_6_7_ok)
print("4/5 near-duplicate confirmed:", duplicate_4_5_ok)

inventory_relationship_gate = (
    all(checks)
    and duplicate_6_7_ok
    and duplicate_4_5_ok
)

print()
print(
    "BOOK RELATIONSHIP INVENTORY GATE:",
    "PASS" if inventory_relationship_gate else "FAIL",
)

[1] Money and ME By Komeil Roudi 040214.docx
    Work group: money_and_me
    Role: AUTHOR_SOURCE_ENGLISH
    Canonical status: SOURCE

[2] Money_and_Me_FA_Published_Final_RTL_FIXED.docx
    Work group: money_and_me
    Role: USER_CREATED_PERSIAN_TRANSLATION
    Canonical status: TRANSLATION_QA_REQUIRED

[4] سواد مالی برای همه.docx
    Work group: financial_literacy_for_all
    Role: AUTHOR_SOURCE
    Canonical status: SOURCE

[5] سواد_مالی_برای_همه_نسخه_نهایی_بازسازی_کامل.docx
    Work group: financial_literacy_for_all
    Role: USER_REVISED_VERSION
    Canonical status: NON_CANONICAL

[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
    Work group: trainer_training_text
    Role: AUTHOR_SOURCE
    Canonical status: SOURCE

[7] نوشتار_آموزشی_تربیت_مربی_سواد_مالی_نسخه_ورد_بازطراحی_RTL.docx
    Work group: trainer_training_text
    Role: USER_REVISED_VERSION
    Canonical status: NON_CANONICAL

RELATIONSHIP VALIDATION
------------------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی هویت فایل باقی‌مانده در Inventory

روابط شناخته‌شده بین فایل‌های `[1]`، `[2]`، `[4]`، `[5]`، `[6]` و `[7]` با موفقیت ثبت و Validation شدند:

`BOOK RELATIONSHIP INVENTORY GATE: PASS`

در حال حاضر تنها فایل بدون Work Group تعیین‌شده، فایل `[3]` است:

`درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx`

در این مرحله فقط ساختار و Metadata داخلی همین فایل بررسی می‌شود تا بتوانیم درباره هویت آن بر اساس شواهد واقعی تصمیم بگیریم.

موارد بررسی‌شده شامل:

- Metadata داخلی `DOCX`
- تعداد Paragraphهای دارای متن
- نسبت حروف فارسی و لاتین
- Paragraphهای ابتدایی
- Headingهای ثبت‌شده در ساختار Word
- Styleهای پرکاربرد Paragraphها

این مرحله فقط `Inspection` است و هنوز هیچ `Work ID` یا وضعیت Canonical ایجاد نمی‌کند.

**معیار PASS:**

فایل `[3]` باید قابل‌خواندن باشد و اطلاعات کافی برای بررسی انسانی هویت و ساختار آن استخراج شود.

</div>

In [4]:
# Code Cell 04

import re
import zipfile
from collections import Counter
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
CP_NS = "http://schemas.openxmlformats.org/package/2006/metadata/core-properties"
DC_NS = "http://purl.org/dc/elements/1.1/"
DCT_NS = "http://purl.org/dc/terms/"

target_path = book_files[2]

with zipfile.ZipFile(target_path) as archive:
    document_root = ET.fromstring(archive.read("word/document.xml"))

    metadata = {}

    if "docProps/core.xml" in archive.namelist():
        core_root = ET.fromstring(archive.read("docProps/core.xml"))

        metadata_fields = {
            "title": f"{{{DC_NS}}}title",
            "creator": f"{{{DC_NS}}}creator",
            "last_modified_by": f"{{{CP_NS}}}lastModifiedBy",
            "created": f"{{{DCT_NS}}}created",
            "modified": f"{{{DCT_NS}}}modified",
        }

        for key, tag in metadata_fields.items():
            node = core_root.find(tag)
            metadata[key] = (
                node.text.strip()
                if node is not None and node.text
                else None
            )

paragraphs = []
headings = []
styles = []

for paragraph in document_root.findall(f".//{{{W_NS}}}p"):
    text = "".join(
        node.text or ""
        for node in paragraph.findall(f".//{{{W_NS}}}t")
    ).strip()

    if not text:
        continue

    style = None
    ppr = paragraph.find(f"{{{W_NS}}}pPr")

    if ppr is not None:
        style_node = ppr.find(f"{{{W_NS}}}pStyle")
        if style_node is not None:
            style = style_node.get(f"{{{W_NS}}}val")

    paragraphs.append(text)

    if style:
        styles.append(style)

        if "heading" in style.casefold():
            headings.append((style, text))

full_text = " ".join(paragraphs)

persian_chars = len(re.findall(r"[\u0600-\u06FF]", full_text))
latin_chars = len(re.findall(r"[A-Za-z]", full_text))
language_total = persian_chars + latin_chars

persian_ratio = persian_chars / language_total if language_total else 0
latin_ratio = latin_chars / language_total if language_total else 0

print("FILE [3] IDENTITY INSPECTION")
print("=" * 90)
print("File:", target_path.name)
print("Metadata:", metadata)
print("Meaningful paragraphs:", len(paragraphs))
print(f"Persian character ratio: {persian_ratio:.3f}")
print(f"Latin character ratio:   {latin_ratio:.3f}")

print("\nFirst meaningful paragraphs:")
for paragraph in paragraphs[:12]:
    print("-", paragraph[:300])

print("\nDetected headings:", len(headings))
for style, heading in headings[:20]:
    print(f"- [{style}] {heading[:300]}")

print("\nMost common paragraph styles:")
for style, count in Counter(styles).most_common(10):
    print(f"- {style}: {count}")

inspection_pass = bool(paragraphs) and persian_ratio > 0.5

print()
print(
    "FILE [3] IDENTITY INSPECTION GATE:",
    "PASS" if inspection_pass else "FAIL",
)

FILE [3] IDENTITY INSPECTION
File: درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Metadata: {'title': None, 'creator': 'sajad yazdani', 'last_modified_by': 'ASUS', 'created': '2020-07-14T21:09:00Z', 'modified': '2021-08-21T08:15:00Z'}
Meaningful paragraphs: 1333
Persian character ratio: 0.978
Latin character ratio:   0.022

First meaningful paragraphs:
- نوشتار آموزشی دورهٔ آموزش سواد مالی بزرگسالان
- دکتر کمیل رودی
- فهرست
- سخن آکادمی هوش مالی1
- فصل ا: اهمیت و تعریف سواد مالی3
- درس ۱: جایگاه سواد مالی در توسعهٔ فردی3
- درس ۲: پلکان توانمندی مالی4
- خودارزیابی 1: موقعیت من در پلکان توانمندی مالی5
- کاربرگ 1: نامه‌ای به خودتان5
- کاربرگ 2: خودآزمایی مالی5
- فصل 2: خرج و پس‌انداز8
- درس ۳: نگرانی یا مسأله مالی8

Detected headings: 98
- [Heading1] سخن آکادمی هوش مالی
- [Heading1] فصل ا: اهمیت و تعریف سواد مالی
- [Heading1] درس ۱: جایگاه سواد مالی در توسعهٔ فردی
- [Heading1] درس ۲: پلکان توانمندی مالی
- [Heading2] خودارزیابی 1: موقعیت من در پلکان توانمندی مالی
- [Heading2] کاربرگ 1:

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تثبیت اولیه گروه‌های Work بر اساس شواهد فایل و اطلاعات منبع

بررسی فایل `[3]` با موفقیت انجام شد:

`FILE [3] IDENTITY INSPECTION GATE: PASS`

اطلاعات منبع نیز مشخص می‌کند که فایل `[3]` یکی از نوشتارهای اصلی دریافت‌شده از مولف است و محتوای آموزشی ویدیویی مرتبط بعداً بر اساس این منبع تولید شده است.

بنابراین در وضعیت فعلی، ۷ فایل فیزیکی به‌صورت اولیه در چهار گروه محتوایی قرار می‌گیرند:

1. `Money and Me`
   - `[1]` نسخه انگلیسی اصلی مولف
   - `[2]` ترجمه فارسی تهیه‌شده از نسخه اصلی و نیازمند QA

2. `نوشتار آموزشی دوره آموزش سواد مالی بزرگسالان`
   - `[3]` منبع اصلی مولف
   - مبنای محتوایی برای بخشی از دوره‌های ویدیویی مرتبط

3. `سواد مالی برای همه`
   - `[4]` منبع اصلی مولف
   - `[5]` نسخه بازسازی‌شده توسط کاربر و غیر Canonical

4. `نوشتار آموزشی تربیت مربی سواد مالی`
   - `[6]` منبع اصلی مولف
   - `[7]` نسخه بازطراحی‌شده با متن یکسان و غیر Canonical

این چهار گروه هنوز `Work ID` رسمی نیستند؛ در این مرحله فقط Grouping اولیه Inventory ثبت می‌شود.

هیچ فایل فیزیکی حذف نمی‌شود تا Lineage و Audit Trail حفظ شود.

**معیار PASS:**

- هر ۷ فایل دقیقاً به یک گروه Inventory تعلق داشته باشد.
- فایل‌های اصلی مولف از نسخه‌های ترجمه‌شده یا بازسازی‌شده جدا باشند.
- تعداد گروه‌های اولیه بدون Duplicate برابر با ۴ باشد.

</div>

In [5]:
# Code Cell 05

inventory_records = [
    {
        "file_index": 1,
        "work_group": "money_and_me",
        "role": "AUTHOR_SOURCE_ENGLISH",
        "status": "SOURCE",
    },
    {
        "file_index": 2,
        "work_group": "money_and_me",
        "role": "USER_CREATED_PERSIAN_TRANSLATION",
        "status": "TRANSLATION_QA_REQUIRED",
    },
    {
        "file_index": 3,
        "work_group": "adult_financial_literacy_course_text",
        "role": "AUTHOR_SOURCE",
        "status": "SOURCE",
    },
    {
        "file_index": 4,
        "work_group": "financial_literacy_for_all",
        "role": "AUTHOR_SOURCE",
        "status": "SOURCE",
    },
    {
        "file_index": 5,
        "work_group": "financial_literacy_for_all",
        "role": "USER_REVISED_VERSION",
        "status": "NON_CANONICAL",
    },
    {
        "file_index": 6,
        "work_group": "trainer_training_text",
        "role": "AUTHOR_SOURCE",
        "status": "SOURCE",
    },
    {
        "file_index": 7,
        "work_group": "trainer_training_text",
        "role": "USER_REVISED_VERSION",
        "status": "NON_CANONICAL",
    },
]

assigned_indices = [record["file_index"] for record in inventory_records]
work_groups = sorted({record["work_group"] for record in inventory_records})

for record in inventory_records:
    index = record["file_index"]

    print(f"[{index}] {book_files[index - 1].name}")
    print("    Work group:", record["work_group"])
    print("    Role:", record["role"])
    print("    Status:", record["status"])
    print()

print("INVENTORY SUMMARY")
print("-" * 80)
print("Physical files:", len(book_files))
print("Assigned files:", len(assigned_indices))
print("Unique assigned files:", len(set(assigned_indices)))
print("Provisional work groups:", len(work_groups))

for group in work_groups:
    members = [
        record["file_index"]
        for record in inventory_records
        if record["work_group"] == group
    ]
    print(f"- {group}: {members}")

inventory_grouping_pass = (
    len(assigned_indices) == len(book_files)
    and len(set(assigned_indices)) == len(book_files)
    and set(assigned_indices) == set(range(1, len(book_files) + 1))
    and len(work_groups) == 4
)

print()
print(
    "PROVISIONAL WORK GROUPING GATE:",
    "PASS" if inventory_grouping_pass else "FAIL",
)

[1] Money and ME By Komeil Roudi 040214.docx
    Work group: money_and_me
    Role: AUTHOR_SOURCE_ENGLISH
    Status: SOURCE

[2] Money_and_Me_FA_Published_Final_RTL_FIXED.docx
    Work group: money_and_me
    Role: USER_CREATED_PERSIAN_TRANSLATION
    Status: TRANSLATION_QA_REQUIRED

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
    Work group: adult_financial_literacy_course_text
    Role: AUTHOR_SOURCE
    Status: SOURCE

[4] سواد مالی برای همه.docx
    Work group: financial_literacy_for_all
    Role: AUTHOR_SOURCE
    Status: SOURCE

[5] سواد_مالی_برای_همه_نسخه_نهایی_بازسازی_کامل.docx
    Work group: financial_literacy_for_all
    Role: USER_REVISED_VERSION
    Status: NON_CANONICAL

[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
    Work group: trainer_training_text
    Role: AUTHOR_SOURCE
    Status: SOURCE

[7] نوشتار_آموزشی_تربیت_مربی_سواد_مالی_نسخه_ورد_بازطراحی_RTL.docx
    Work group: trainer_training_text
    Role: USER_REVISED_VERSION
    Status: NON_CANONICAL

INVE

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تعریف مدل رسمی Work / Edition / Physical File و شناسه‌های پایدار

Grouping اولیه با موفقیت تأیید شد:

`PROVISIONAL WORK GROUPING GATE: PASS`

در این مرحله مدل هویتی کتاب‌ها رسمی‌تر می‌شود.

سه سطح مستقل در نظر گرفته می‌شود:

`Work`
اثر فکری مستقل، بدون وابستگی به یک فایل یا قالب خاص.

`Edition`
یک نسخه مشخص از Work؛ برای مثال نسخه انگلیسی اصلی، ترجمه فارسی، یا نسخه بازسازی‌شده.

`Physical File`
فایل واقعی `DOCX` موجود در `data/books/`.

بر اساس شواهد فعلی چهار Work داریم.

برای اثر `Money and Me` باید دو مفهوم جدا بمانند:

- نسخه انگلیسی `[1]` منبع اصلی مولف و Source of Truth است.
- نسخه فارسی `[2]` ترجمه تهیه‌شده از آن است و به دلیل فارسی بودن RAG می‌تواند Candidate مناسب برای Ingestion باشد، اما پذیرش نهایی آن منوط به QA ترجمه است.

برای سه Work دیگر، نسخه اصلی دریافت‌شده از مولف به‌عنوان Source Edition ثبت می‌شود.

فایل‌های `[5]` و `[7]` حذف یا نادیده گرفته نمی‌شوند؛ در Inventory باقی می‌مانند ولی `NON_CANONICAL` هستند تا Lineage حفظ شود.

در این مرحله شناسه‌های پایدار و قابل‌خواندن تعریف می‌شوند، اما هنوز Manifest نهایی روی دیسک نوشته نمی‌شود.

**معیار PASS:**

- دقیقاً ۴ `Work ID` یکتا وجود داشته باشد.
- هر ۷ فایل دقیقاً به یک `Edition ID` متصل باشد.
- هیچ `Edition ID` تکراری وجود نداشته باشد.
- Source Edition هر Work مشخص باشد.
- وضعیت خاص ترجمه فارسی `Money and Me` به‌صورت `QA_REQUIRED` حفظ شود.

</div>

In [6]:
# Code Cell 06

work_records = [
    {
        "work_id": "book:money-and-me",
        "title": "Money and Me",
        "source_edition_id": "book:money-and-me:en-source",
    },
    {
        "work_id": "book:adult-financial-literacy-course",
        "title": "نوشتار آموزشی دوره آموزش سواد مالی بزرگسالان",
        "source_edition_id": "book:adult-financial-literacy-course:fa-source",
    },
    {
        "work_id": "book:financial-literacy-for-all",
        "title": "سواد مالی برای همه",
        "source_edition_id": "book:financial-literacy-for-all:fa-source",
    },
    {
        "work_id": "book:trainer-training-financial-literacy",
        "title": "نوشتار آموزشی تربیت مربی سواد مالی",
        "source_edition_id": "book:trainer-training-financial-literacy:fa-source",
    },
]

edition_records = [
    {
        "edition_id": "book:money-and-me:en-source",
        "work_id": "book:money-and-me",
        "file_index": 1,
        "language": "en",
        "edition_role": "AUTHOR_SOURCE",
        "ingestion_status": "SOURCE_OF_TRUTH",
    },
    {
        "edition_id": "book:money-and-me:fa-user-translation",
        "work_id": "book:money-and-me",
        "file_index": 2,
        "language": "fa",
        "edition_role": "USER_TRANSLATION",
        "ingestion_status": "QA_REQUIRED",
    },
    {
        "edition_id": "book:adult-financial-literacy-course:fa-source",
        "work_id": "book:adult-financial-literacy-course",
        "file_index": 3,
        "language": "fa",
        "edition_role": "AUTHOR_SOURCE",
        "ingestion_status": "CANONICAL_SOURCE",
    },
    {
        "edition_id": "book:financial-literacy-for-all:fa-source",
        "work_id": "book:financial-literacy-for-all",
        "file_index": 4,
        "language": "fa",
        "edition_role": "AUTHOR_SOURCE",
        "ingestion_status": "CANONICAL_SOURCE",
    },
    {
        "edition_id": "book:financial-literacy-for-all:fa-user-revised",
        "work_id": "book:financial-literacy-for-all",
        "file_index": 5,
        "language": "fa",
        "edition_role": "USER_REVISED",
        "ingestion_status": "NON_CANONICAL",
    },
    {
        "edition_id": "book:trainer-training-financial-literacy:fa-source",
        "work_id": "book:trainer-training-financial-literacy",
        "file_index": 6,
        "language": "fa",
        "edition_role": "AUTHOR_SOURCE",
        "ingestion_status": "CANONICAL_SOURCE",
    },
    {
        "edition_id": "book:trainer-training-financial-literacy:fa-user-revised",
        "work_id": "book:trainer-training-financial-literacy",
        "file_index": 7,
        "language": "fa",
        "edition_role": "USER_REVISED",
        "ingestion_status": "NON_CANONICAL",
    },
]

work_ids = [record["work_id"] for record in work_records]
edition_ids = [record["edition_id"] for record in edition_records]
file_indices = [record["file_index"] for record in edition_records]

valid_work_ids = set(work_ids)
edition_work_ids = {record["work_id"] for record in edition_records}

source_editions_exist = all(
    work["source_edition_id"] in set(edition_ids)
    for work in work_records
)

print("WORK MODEL")
print("-" * 90)

for work in work_records:
    members = [
        edition
        for edition in edition_records
        if edition["work_id"] == work["work_id"]
    ]

    print(work["work_id"])
    print("    Title:", work["title"])
    print("    Source edition:", work["source_edition_id"])

    for edition in members:
        print(
            f"    - {edition['edition_id']} "
            f"| file [{edition['file_index']}] "
            f"| {edition['edition_role']} "
            f"| {edition['ingestion_status']}"
        )

    print()

model_pass = (
    len(work_records) == 4
    and len(work_ids) == len(set(work_ids))
    and len(edition_records) == 7
    and len(edition_ids) == len(set(edition_ids))
    and set(file_indices) == set(range(1, 8))
    and edition_work_ids == valid_work_ids
    and source_editions_exist
)

print("Works:", len(work_records))
print("Editions:", len(edition_records))
print("Physical files mapped:", len(set(file_indices)))
print("Unique Work IDs:", len(set(work_ids)))
print("Unique Edition IDs:", len(set(edition_ids)))
print()
print(
    "WORK / EDITION / FILE MODEL GATE:",
    "PASS" if model_pass else "FAIL",
)

WORK MODEL
------------------------------------------------------------------------------------------
book:money-and-me
    Title: Money and Me
    Source edition: book:money-and-me:en-source
    - book:money-and-me:en-source | file [1] | AUTHOR_SOURCE | SOURCE_OF_TRUTH
    - book:money-and-me:fa-user-translation | file [2] | USER_TRANSLATION | QA_REQUIRED

book:adult-financial-literacy-course
    Title: نوشتار آموزشی دوره آموزش سواد مالی بزرگسالان
    Source edition: book:adult-financial-literacy-course:fa-source
    - book:adult-financial-literacy-course:fa-source | file [3] | AUTHOR_SOURCE | CANONICAL_SOURCE

book:financial-literacy-for-all
    Title: سواد مالی برای همه
    Source edition: book:financial-literacy-for-all:fa-source
    - book:financial-literacy-for-all:fa-source | file [4] | AUTHOR_SOURCE | CANONICAL_SOURCE
    - book:financial-literacy-for-all:fa-user-revised | file [5] | USER_REVISED | NON_CANONICAL

book:trainer-training-financial-literacy
    Title: نوشتار آموزشی

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی شواهد نویسندگی، انتساب و وضعیت حقوق منابع اصلی

مدل `Work / Edition / File` با موفقیت تأیید شد:

`WORK / EDITION / FILE MODEL GATE: PASS`

در این مرحله فقط نسخه‌های اصلی یا Source Edition چهار Work بررسی می‌شوند:

- فایل `[1]`
- فایل `[3]`
- فایل `[4]`
- فایل `[6]`

هدف این است که بین مفاهیم زیر تفکیک روشن ایجاد شود:

- `Author / Creator`
- `Knowledge Owner`
- `Rights Holder`
- `Rights Status`
- `Rights Evidence`

طبق اطلاعات انسانی ثبت‌شده در این Notebook، این چهار فایل به‌عنوان منابع اصلی دریافت‌شده از مولف معرفی شده‌اند؛ اما این موضوع به‌تنهایی اثبات‌کننده مالکیت حقوقی یا مجوز استفاده نیست.

بنابراین در این مرحله از خود فایل‌ها نیز شواهد قابل مشاهده برای انتساب نویسنده استخراج می‌شود:

- Metadata داخلی Word
- Paragraphهای ابتدایی
- وجود نام‌هایی مانند `کمیل رودی` یا `Komeil Roudi`

هیچ `rights_status` قطعی بدون Evidence مستقل ساخته نمی‌شود.

**معیار PASS:**

- هر چهار Source Edition قابل بررسی باشد.
- Evidence مربوط به Attribution برای هر فایل نمایش داده شود.
- نبود Evidence حقوقی به‌صورت صریح از Attribution جدا نگه داشته شود.

</div>

In [7]:
# Code Cell 07

import re
import zipfile
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
CP_NS = "http://schemas.openxmlformats.org/package/2006/metadata/core-properties"
DC_NS = "http://purl.org/dc/elements/1.1/"

source_file_indices = [1, 3, 4, 6]

def inspect_attribution_evidence(path):
    with zipfile.ZipFile(path) as archive:
        document_root = ET.fromstring(archive.read("word/document.xml"))

        metadata = {
            "creator": None,
            "last_modified_by": None,
        }

        if "docProps/core.xml" in archive.namelist():
            core_root = ET.fromstring(archive.read("docProps/core.xml"))

            creator_node = core_root.find(f"{{{DC_NS}}}creator")
            modified_by_node = core_root.find(f"{{{CP_NS}}}lastModifiedBy")

            if creator_node is not None and creator_node.text:
                metadata["creator"] = creator_node.text.strip()

            if modified_by_node is not None and modified_by_node.text:
                metadata["last_modified_by"] = modified_by_node.text.strip()

    paragraphs = []

    for paragraph in document_root.findall(f".//{{{W_NS}}}p"):
        text = "".join(
            node.text or ""
            for node in paragraph.findall(f".//{{{W_NS}}}t")
        ).strip()

        if text:
            paragraphs.append(text)

    name_pattern = re.compile(
        r"کمیل\s+رودی|دکتر\s+کمیل\s+رودی|Komeil\s+Roudi",
        flags=re.IGNORECASE,
    )

    name_matches = [
        text
        for text in paragraphs
        if name_pattern.search(text)
    ]

    return {
        "metadata": metadata,
        "paragraphs": paragraphs,
        "name_matches": name_matches,
    }

attribution_results = {}

for file_index in source_file_indices:
    path = book_files[file_index - 1]
    result = inspect_attribution_evidence(path)
    attribution_results[file_index] = result

    print("=" * 90)
    print(f"[{file_index}] {path.name}")
    print("Creator metadata:", result["metadata"]["creator"])
    print("Last modified by:", result["metadata"]["last_modified_by"])
    print("Explicit Komeil Roudi references:", len(result["name_matches"]))

    for text in result["name_matches"][:10]:
        print(" -", text[:300])

    print("\nFirst meaningful paragraphs:")
    for text in result["paragraphs"][:8]:
        print(" -", text[:300])

    print()

attribution_pass = (
    len(attribution_results) == 4
    and all(result["paragraphs"] for result in attribution_results.values())
)

print("ATTRIBUTION EVIDENCE FILES:", len(attribution_results))
print("RIGHTS EVIDENCE STATUS: NOT YET ESTABLISHED")
print()
print(
    "BOOK ATTRIBUTION EVIDENCE GATE:",
    "PASS" if attribution_pass else "FAIL",
)

[1] Money and ME By Komeil Roudi 040214.docx
Creator metadata: ASUS
Last modified by: ASUS
Explicit Komeil Roudi references: 1
 - by: Komeil Roudi

First meaningful paragraphs:
 - Money and Me
 - A Personal Journey to Financial Literacy
 - by: Komeil Roudi
 - Financial Literacy: Misconceptions and Realities
 - The term financial literacy is both appealing and straightforward in its wording. At first glance, its meaning seems obvious: it is the combination of knowledge, skills, attitudes and ultimately behaviours that translate into sound financial decisions. However, this phrase has different interpretati
 - Financial Literacy vs. Wealth
 - Some discussions labelled as financial literacy are centred around wealth creation. While wealth and the pursuit of it are legitimate topics, this approach focuses primarily on increasing income and achieving financial success. It encourages individuals to aspire to wealth by showcasing the lives of
 - Financial Literacy vs. Entrepreneurship

[3] در

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تثبیت Attribution مولف و تفکیک آن از Metadata فنی فایل

بررسی Attribution فایل‌های اصلی با موفقیت انجام شد:

`BOOK ATTRIBUTION EVIDENCE GATE: PASS`

بر اساس اطلاعات انسانی تأییدشده درباره منشأ این منابع، هر چهار Work اصلی متعلق به آثار دکتر کمیل رودی هستند و مولف اصلی آن‌ها:

`Dr. Komeil Roudi / دکتر کمیل رودی`

است.

نام‌هایی که در Metadata داخلی برخی فایل‌های Word دیده شدند، مانند:

- `ASUS`
- `sajad yazdani`
- `faty gh`

به‌عنوان شواهد نویسندگی استفاده نمی‌شوند، زیرا فیلدهایی مانند `creator` و `last_modified_by` در Word می‌توانند مربوط به سازنده فایل، ویراستار، تایپیست یا سیستم تولید فایل باشند.

بنابراین در مدل پروژه:

- `author/creator = Dr. Komeil Roudi`
- `knowledge_owner = Dr. Komeil Roudi`

برای هر چهار Work اصلی ثبت می‌شود.

در نسخه فارسی `Money and Me`، نویسنده اثر همچنان دکتر کمیل رودی است ولی نقش ترجمه باید جداگانه به‌عنوان `USER_TRANSLATION` حفظ شود.

همچنین نسخه‌های `[5]` و `[7]` اثر جدید محسوب نمی‌شوند و نقش بازسازی/بازطراحی کاربر باید جدا از نویسندگی ثبت شود.

نکته مهم:

`Authorship != Rights`

تأیید مولف به‌تنهایی اثبات‌کننده `rights_holder`، `allowed_uses` یا وضعیت حقوقی استفاده در محصول نیست. این بخش تا زمان وجود Evidence مستقل باز باقی می‌ماند.

**معیار PASS:**

- هر چهار Work دارای مولف و `knowledge_owner` مشخص باشند.
- Metadata فنی Word به‌عنوان نویسنده ثبت نشود.
- نقش ترجمه و بازسازی با Authorship مخلوط نشود.
- Rights همچنان مستقل و بدون حدس باقی بماند.

</div>

In [8]:
# Code Cell 08

work_attribution_records = [
    {
        "work_id": "book:money-and-me",
        "author": "Dr. Komeil Roudi",
        "knowledge_owner": "Dr. Komeil Roudi",
        "attribution_basis": "HUMAN_CONFIRMED_AUTHORSHIP",
        "rights_status": "EVIDENCE_REQUIRED",
    },
    {
        "work_id": "book:adult-financial-literacy-course",
        "author": "Dr. Komeil Roudi",
        "knowledge_owner": "Dr. Komeil Roudi",
        "attribution_basis": "HUMAN_CONFIRMED_AUTHORSHIP",
        "rights_status": "EVIDENCE_REQUIRED",
    },
    {
        "work_id": "book:financial-literacy-for-all",
        "author": "Dr. Komeil Roudi",
        "knowledge_owner": "Dr. Komeil Roudi",
        "attribution_basis": "HUMAN_CONFIRMED_AUTHORSHIP",
        "rights_status": "EVIDENCE_REQUIRED",
    },
    {
        "work_id": "book:trainer-training-financial-literacy",
        "author": "Dr. Komeil Roudi",
        "knowledge_owner": "Dr. Komeil Roudi",
        "attribution_basis": "HUMAN_CONFIRMED_AUTHORSHIP",
        "rights_status": "EVIDENCE_REQUIRED",
    },
]

known_work_ids = {record["work_id"] for record in work_records}
attribution_work_ids = {
    record["work_id"]
    for record in work_attribution_records
}

for record in work_attribution_records:
    print(record["work_id"])
    print("    Author:", record["author"])
    print("    Knowledge owner:", record["knowledge_owner"])
    print("    Attribution basis:", record["attribution_basis"])
    print("    Rights status:", record["rights_status"])
    print()

attribution_model_pass = (
    len(work_attribution_records) == 4
    and attribution_work_ids == known_work_ids
    and all(
        record["author"] == "Dr. Komeil Roudi"
        and record["knowledge_owner"] == "Dr. Komeil Roudi"
        and record["rights_status"] == "EVIDENCE_REQUIRED"
        for record in work_attribution_records
    )
)

print(
    "WORK ATTRIBUTION MODEL GATE:",
    "PASS" if attribution_model_pass else "FAIL",
)

book:money-and-me
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Attribution basis: HUMAN_CONFIRMED_AUTHORSHIP
    Rights status: EVIDENCE_REQUIRED

book:adult-financial-literacy-course
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Attribution basis: HUMAN_CONFIRMED_AUTHORSHIP
    Rights status: EVIDENCE_REQUIRED

book:financial-literacy-for-all
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Attribution basis: HUMAN_CONFIRMED_AUTHORSHIP
    Rights status: EVIDENCE_REQUIRED

book:trainer-training-financial-literacy
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Attribution basis: HUMAN_CONFIRMED_AUTHORSHIP
    Rights status: EVIDENCE_REQUIRED

WORK ATTRIBUTION MODEL GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تعیین Candidateهای Canonical برای Book Ingestion و موارد Quarantine

مدل Attribution با موفقیت تأیید شد:

`WORK ATTRIBUTION MODEL GATE: PASS`

اکنون برای هر Work باید مشخص شود کدام Physical File در مرحله بعد، یعنی `04_book_ingestion.ipynb`، مجاز به ورود به Pipeline است.

بر اساس شواهد فعلی:

#### `Money and Me`

- فایل `[1]` نسخه انگلیسی اصلی مولف و `Source of Truth` است.
- فایل `[2]` ترجمه فارسی تهیه‌شده از فایل `[1]` است.
- چون محصول RAG فارسی است، فایل `[2]` Candidate مناسب برای Ingestion فارسی محسوب می‌شود.
- اما کیفیت ترجمه هنوز در این Notebook تأیید نشده است.

بنابراین:

`[1] = AUTHORITATIVE_REFERENCE`

`[2] = QUARANTINED_PENDING_TRANSLATION_QA`

تا زمانی که QA ترجمه انجام نشده، فایل `[2]` نباید به‌عنوان نسخه نهایی پذیرفته‌شده وارد Corpus شود.

#### سه Work فارسی دیگر

فایل‌های `[3]`، `[4]` و `[6]` منابع اصلی دریافت‌شده از مولف هستند و Candidateهای Canonical برای Book Ingestion محسوب می‌شوند.

#### نسخه‌های بازسازی‌شده

فایل‌های `[5]` و `[7]` در Inventory و Lineage باقی می‌مانند، اما برای Ingestion انتخاب نمی‌شوند.

`NON_CANONICAL != DELETE`

در این مرحله هیچ فایل فیزیکی حذف نمی‌شود.

**معیار PASS:**

- هر ۷ فایل دارای تصمیم Inventory روشن باشد.
- فقط نسخه‌های اصلی مولف برای سه Work فارسی مستقیماً `ACCEPTED_FOR_INGESTION` باشند.
- ترجمه فارسی `Money and Me` تا انجام QA در Quarantine باقی بماند.
- نسخه‌های بازسازی‌شده وارد Corpus نشوند.

</div>

In [9]:
# Code Cell 09

ingestion_decisions = [
    {
        "file_index": 1,
        "work_id": "book:money-and-me",
        "decision": "AUTHORITATIVE_REFERENCE",
        "reason": "Original English author source",
    },
    {
        "file_index": 2,
        "work_id": "book:money-and-me",
        "decision": "QUARANTINED_PENDING_TRANSLATION_QA",
        "reason": "Persian user translation required for Persian RAG but not yet independently validated",
    },
    {
        "file_index": 3,
        "work_id": "book:adult-financial-literacy-course",
        "decision": "ACCEPTED_FOR_INGESTION",
        "reason": "Original Persian author source",
    },
    {
        "file_index": 4,
        "work_id": "book:financial-literacy-for-all",
        "decision": "ACCEPTED_FOR_INGESTION",
        "reason": "Original Persian author source",
    },
    {
        "file_index": 5,
        "work_id": "book:financial-literacy-for-all",
        "decision": "NON_CANONICAL",
        "reason": "User-revised version of original author source",
    },
    {
        "file_index": 6,
        "work_id": "book:trainer-training-financial-literacy",
        "decision": "ACCEPTED_FOR_INGESTION",
        "reason": "Original Persian author source",
    },
    {
        "file_index": 7,
        "work_id": "book:trainer-training-financial-literacy",
        "decision": "NON_CANONICAL",
        "reason": "User-reformatted version with exact same extracted text",
    },
]

for record in ingestion_decisions:
    index = record["file_index"]

    print(f"[{index}] {book_files[index - 1].name}")
    print("    Work ID:", record["work_id"])
    print("    Decision:", record["decision"])
    print("    Reason:", record["reason"])
    print()

decision_indices = [record["file_index"] for record in ingestion_decisions]

accepted_indices = {
    record["file_index"]
    for record in ingestion_decisions
    if record["decision"] == "ACCEPTED_FOR_INGESTION"
}

quarantined_indices = {
    record["file_index"]
    for record in ingestion_decisions
    if record["decision"].startswith("QUARANTINED")
}

non_canonical_indices = {
    record["file_index"]
    for record in ingestion_decisions
    if record["decision"] == "NON_CANONICAL"
}

decision_gate = (
    len(ingestion_decisions) == 7
    and len(set(decision_indices)) == 7
    and set(decision_indices) == set(range(1, 8))
    and accepted_indices == {3, 4, 6}
    and quarantined_indices == {2}
    and non_canonical_indices == {5, 7}
)

print("INGESTION DECISION SUMMARY")
print("-" * 80)
print("Accepted for ingestion:", sorted(accepted_indices))
print("Quarantined:", sorted(quarantined_indices))
print("Non-canonical:", sorted(non_canonical_indices))
print("Authoritative reference only:", [1])
print()
print(
    "BOOK INGESTION SELECTION GATE:",
    "PASS" if decision_gate else "FAIL",
)

[1] Money and ME By Komeil Roudi 040214.docx
    Work ID: book:money-and-me
    Decision: AUTHORITATIVE_REFERENCE
    Reason: Original English author source

[2] Money_and_Me_FA_Published_Final_RTL_FIXED.docx
    Work ID: book:money-and-me
    Decision: QUARANTINED_PENDING_TRANSLATION_QA
    Reason: Persian user translation required for Persian RAG but not yet independently validated

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
    Work ID: book:adult-financial-literacy-course
    Decision: ACCEPTED_FOR_INGESTION
    Reason: Original Persian author source

[4] سواد مالی برای همه.docx
    Work ID: book:financial-literacy-for-all
    Decision: ACCEPTED_FOR_INGESTION
    Reason: Original Persian author source

[5] سواد_مالی_برای_همه_نسخه_نهایی_بازسازی_کامل.docx
    Work ID: book:financial-literacy-for-all
    Decision: NON_CANONICAL
    Reason: User-revised version of original author source

[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
    Work ID: book:trainer-training-financi

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی ساختار داخلی DOCX برای طراحی Citation پایدار

انتخاب منابع مرحله بعد با موفقیت تأیید شد:

`BOOK INGESTION SELECTION GATE: PASS`

اکنون باید مشخص شود ساختار واقعی فایل‌های کتاب تا چه حد برای Citation پایدار قابل استفاده است.

در فایل‌های `DOCX` نباید به شماره صفحه به‌عنوان Locator اصلی اعتماد کنیم، زیرا Pagination می‌تواند با تغییر Word، فونت، اندازه صفحه یا Rendering تغییر کند.

ساختار پیشنهادی پروژه برای Citation کتاب‌ها در آینده به‌صورت زیر است:

`Work ID → Edition ID → Heading Path → Stable Paragraph ID`

برای تصمیم‌گیری درباره امکان استفاده از این مدل، در این مرحله ساختار نسخه‌های اصلی و نسخه فارسی Candidate بررسی می‌شود:

- `[1]` نسخه انگلیسی اصلی `Money and Me`
- `[2]` ترجمه فارسی Candidate
- `[3]` منبع اصلی فارسی
- `[4]` منبع اصلی فارسی
- `[6]` منبع اصلی فارسی

برای هر فایل موارد زیر اندازه‌گیری می‌شود:

- تعداد Paragraphهای دارای متن
- تعداد Headingها
- توزیع Heading Levelها
- تعداد Tableها
- تعداد Paragraphهای داخل Table
- Styleهای اصلی استفاده‌شده
- چند Heading ابتدایی برای بررسی انسانی

فایل‌های `[5]` و `[7]` به دلیل `NON_CANONICAL` بودن در طراحی Citation فعال بررسی نمی‌شوند.

این مرحله فقط `Structural Inspection` است؛ هنوز Paragraph ID ساخته نمی‌شود و هیچ Chunking انجام نمی‌شود.

**معیار PASS:**

- هر پنج Edition موردنظر قابل خواندن باشد.
- ساختار Paragraph برای همه قابل استخراج باشد.
- وضعیت Heading و Table هر فایل به‌صورت واقعی مشخص شود.
- بتوانیم بر اساس خروجی تصمیم بگیریم که آیا `Heading Path + Stable Paragraph ID` مبنای مناسبی برای Citation است یا نیاز به Strategy جایگزین داریم.

</div>

In [10]:
# Code Cell 10

import zipfile
from collections import Counter
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"

citation_candidate_indices = [1, 2, 3, 4, 6]

def inspect_docx_structure(path):
    with zipfile.ZipFile(path) as archive:
        root = ET.fromstring(archive.read("word/document.xml"))

    paragraphs = []
    style_counts = Counter()
    heading_counts = Counter()
    heading_samples = []
    table_paragraph_count = 0

    body = root.find(f"{{{W_NS}}}body")
    tables = body.findall(f".//{{{W_NS}}}tbl") if body is not None else []

    table_paragraph_nodes = set()

    for table in tables:
        for paragraph in table.findall(f".//{{{W_NS}}}p"):
            table_paragraph_nodes.add(id(paragraph))

    for paragraph in root.findall(f".//{{{W_NS}}}p"):
        text = "".join(
            node.text or ""
            for node in paragraph.findall(f".//{{{W_NS}}}t")
        ).strip()

        if not text:
            continue

        style = "NO_STYLE"
        ppr = paragraph.find(f"{{{W_NS}}}pPr")

        if ppr is not None:
            style_node = ppr.find(f"{{{W_NS}}}pStyle")
            if style_node is not None:
                style_value = style_node.get(f"{{{W_NS}}}val")
                if style_value:
                    style = style_value

        paragraphs.append(text)
        style_counts[style] += 1

        if id(paragraph) in table_paragraph_nodes:
            table_paragraph_count += 1

        if "heading" in style.casefold():
            heading_counts[style] += 1

            if len(heading_samples) < 15:
                heading_samples.append((style, text))

    return {
        "paragraph_count": len(paragraphs),
        "table_count": len(tables),
        "table_paragraph_count": table_paragraph_count,
        "style_counts": style_counts,
        "heading_counts": heading_counts,
        "heading_samples": heading_samples,
    }

structure_results = {}

for file_index in citation_candidate_indices:
    path = book_files[file_index - 1]
    result = inspect_docx_structure(path)
    structure_results[file_index] = result

    print("=" * 100)
    print(f"[{file_index}] {path.name}")
    print("Meaningful paragraphs:", result["paragraph_count"])
    print("Tables:", result["table_count"])
    print("Paragraphs inside tables:", result["table_paragraph_count"])
    print(
        "Detected headings:",
        sum(result["heading_counts"].values())
    )

    print("\nHeading distribution:")
    if result["heading_counts"]:
        for style, count in result["heading_counts"].most_common():
            print(f" - {style}: {count}")
    else:
        print(" - NONE")

    print("\nMost common paragraph styles:")
    for style, count in result["style_counts"].most_common(8):
        print(f" - {style}: {count}")

    print("\nFirst detected headings:")
    if result["heading_samples"]:
        for style, text in result["heading_samples"]:
            print(f" - [{style}] {text[:250]}")
    else:
        print(" - NONE")

    print()

structure_gate = (
    len(structure_results) == len(citation_candidate_indices)
    and all(
        result["paragraph_count"] > 0
        for result in structure_results.values()
    )
)

print(
    "BOOK CITATION STRUCTURE INSPECTION GATE:",
    "PASS" if structure_gate else "FAIL",
)

[1] Money and ME By Komeil Roudi 040214.docx
Meaningful paragraphs: 300
Tables: 6
Paragraphs inside tables: 63
Detected headings: 19

Heading distribution:
 - Heading1: 19

Most common paragraph styles:
 - NO_STYLE: 209
 - ListParagraph: 72
 - Heading1: 19

First detected headings:
 - [Heading1] Financial Literacy: Misconceptions and Realities
 - [Heading1] Financial Literacy: A Holistic Understanding
 - [Heading1] The Financial Capability Stairs
 - [Heading1] Turn confusing concerns into solvable problems.
 - [Heading1] Managing expenses by writing them down
 - [Heading1] Be Aware of Purchasing Pitfalls
 - [Heading1] Pay Yourself First, then budget
 - [Heading1] Do not use future income to meet present needs
 - [Heading1] Make Debt Work for You
 - [Heading1] The Importance of Purposeful Work
 - [Heading1] When to Leave Your Job and How to Do It Wisely
 - [Heading1] Creating Value, Not Just Earning Money
 - [Heading1] Making Income Independent of Work
 - [Heading1] Understanding Where 

<div style="direction: rtl; white-space: normal; line-height: 1;">

### طراحی و آزمایش مدل Citation بر پایه Stable Paragraph ID

بررسی ساختار داخلی کتاب‌ها با موفقیت انجام شد:

`BOOK CITATION STRUCTURE INSPECTION GATE: PASS`

نتایج نشان داد ساختار Heading در کتاب‌ها یکدست نیست:

- بعضی فایل‌ها مانند `[2]` و `[3]` Headingهای نسبتاً غنی دارند.
- فایل `[4]` با وجود `871` Paragraph فقط `9` Heading دارد.
- بخش قابل‌توجهی از محتوای بعضی فایل‌ها داخل Table قرار دارد.
- بنابراین `Heading Path` به‌تنهایی نمی‌تواند Locator اصلی Citation باشد.

مدل پیشنهادی Citation کتاب‌ها:

`Work ID → Edition ID → Stable Paragraph ID`

و در صورت وجود:

`Heading Path`

به‌عنوان Context ساختاری افزوده می‌شود.

در این مرحله برای هر Paragraph معنی‌دار یک Locator آزمایشی ساخته می‌شود که شامل موارد زیر است:

- `edition_id`
- شماره ترتیبی Paragraph در همان Edition
- `paragraph_id`
- `text_sha256`
- Style
- Heading Path
- وضعیت قرارگیری داخل Table

شناسه Paragraph بر اساس Edition ثابت و ترتیب Document ساخته می‌شود؛ بنابراین معنای آن وابسته به یک Snapshot مشخص از همان Edition است. `text_sha256` نیز برای Validation محتوا نگهداری می‌شود.

در این مرحله هیچ Artifact نهایی ذخیره نمی‌شود و هیچ Chunking انجام نمی‌شود.

همچنین بررسی می‌کنیم که آیا متن‌های تکراری وجود دارند؛ چون در صورت وجود، Hash متن به‌تنهایی نمی‌تواند Paragraph ID باشد.

**معیار PASS:**

- برای تمام Paragraphهای معنی‌دار پنج Edition مورد بررسی، Paragraph ID یکتا ساخته شود.
- تعداد Locatorها دقیقاً با تعداد Paragraphهای استخراج‌شده برابر باشد.
- هیچ `paragraph_id` تکراری وجود نداشته باشد.
- وضعیت Heading Path و Duplicate Text به‌صورت واقعی اندازه‌گیری شود.

</div>

In [11]:
# Code Cell 11

import hashlib
import re
import zipfile
from collections import Counter
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"

candidate_indices = [1, 2, 3, 4, 6]

edition_id_by_file = {
    record["file_index"]: record["edition_id"]
    for record in edition_records
}

def heading_level_from_style(style):
    if not style:
        return None

    match = re.fullmatch(r"Heading(\d+)", style, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def build_paragraph_locators(path, edition_id):
    with zipfile.ZipFile(path) as archive:
        root = ET.fromstring(archive.read("word/document.xml"))

    table_paragraph_nodes = set()

    for table in root.findall(f".//{{{W_NS}}}tbl"):
        for paragraph in table.findall(f".//{{{W_NS}}}p"):
            table_paragraph_nodes.add(id(paragraph))

    locators = []
    heading_stack = {}

    for paragraph in root.findall(f".//{{{W_NS}}}p"):
        text = "".join(
            node.text or ""
            for node in paragraph.findall(f".//{{{W_NS}}}t")
        ).strip()

        if not text:
            continue

        style = None
        ppr = paragraph.find(f"{{{W_NS}}}pPr")

        if ppr is not None:
            style_node = ppr.find(f"{{{W_NS}}}pStyle")
            if style_node is not None:
                style = style_node.get(f"{{{W_NS}}}val")

        level = heading_level_from_style(style)

        if level is not None:
            heading_stack[level] = text

            for deeper_level in list(heading_stack):
                if deeper_level > level:
                    del heading_stack[deeper_level]

        heading_path = [
            heading_stack[key]
            for key in sorted(heading_stack)
        ]

        paragraph_number = len(locators) + 1
        paragraph_id = f"{edition_id}:p{paragraph_number:06d}"

        text_sha256 = hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest()

        locators.append({
            "paragraph_id": paragraph_id,
            "paragraph_number": paragraph_number,
            "edition_id": edition_id,
            "style": style,
            "heading_path": heading_path,
            "is_heading": level is not None,
            "in_table": id(paragraph) in table_paragraph_nodes,
            "text_sha256": text_sha256,
            "text": text,
        })

    return locators

citation_locator_prototypes = {}

for file_index in candidate_indices:
    edition_id = edition_id_by_file[file_index]
    path = book_files[file_index - 1]

    locators = build_paragraph_locators(path, edition_id)
    citation_locator_prototypes[file_index] = locators

    paragraph_ids = [item["paragraph_id"] for item in locators]
    text_hashes = [item["text_sha256"] for item in locators]

    duplicate_text_members = sum(
        count
        for count in Counter(text_hashes).values()
        if count > 1
    )

    with_heading_path = sum(
        bool(item["heading_path"])
        for item in locators
    )

    table_paragraphs = sum(
        item["in_table"]
        for item in locators
    )

    print("=" * 100)
    print(f"[{file_index}] {path.name}")
    print("Edition ID:", edition_id)
    print("Paragraph locators:", len(locators))
    print("Unique paragraph IDs:", len(set(paragraph_ids)))
    print("Paragraphs with heading path:", with_heading_path)
    print("Paragraphs without heading path:", len(locators) - with_heading_path)
    print("Paragraphs inside tables:", table_paragraphs)
    print("Duplicate-text member paragraphs:", duplicate_text_members)

    print("\nFirst locator samples:")
    for item in locators[:5]:
        print(" -", item["paragraph_id"])
        print("   Heading path:", " > ".join(item["heading_path"]) or "(none)")
        print("   Text:", item["text"][:180])

    print()

all_locators = [
    locator
    for locators in citation_locator_prototypes.values()
    for locator in locators
]

all_paragraph_ids = [
    locator["paragraph_id"]
    for locator in all_locators
]

expected_total = sum(
    structure_results[index]["paragraph_count"]
    for index in candidate_indices
)

citation_model_pass = (
    len(all_locators) == expected_total
    and len(all_paragraph_ids) == len(set(all_paragraph_ids))
    and all(locator["text_sha256"] for locator in all_locators)
)

print("CITATION LOCATOR PROTOTYPE SUMMARY")
print("-" * 80)
print("Expected paragraphs:", expected_total)
print("Locator records:", len(all_locators))
print("Unique paragraph IDs:", len(set(all_paragraph_ids)))
print()
print(
    "BOOK PARAGRAPH LOCATOR PROTOTYPE GATE:",
    "PASS" if citation_model_pass else "FAIL",
)

[1] Money and ME By Komeil Roudi 040214.docx
Edition ID: book:money-and-me:en-source
Paragraph locators: 300
Unique paragraph IDs: 300
Paragraphs with heading path: 297
Paragraphs without heading path: 3
Paragraphs inside tables: 63
Duplicate-text member paragraphs: 30

First locator samples:
 - book:money-and-me:en-source:p000001
   Heading path: (none)
   Text: Money and Me
 - book:money-and-me:en-source:p000002
   Heading path: (none)
   Text: A Personal Journey to Financial Literacy
 - book:money-and-me:en-source:p000003
   Heading path: (none)
   Text: by: Komeil Roudi
 - book:money-and-me:en-source:p000004
   Heading path: Financial Literacy: Misconceptions and Realities
   Text: Financial Literacy: Misconceptions and Realities
 - book:money-and-me:en-source:p000005
   Heading path: Financial Literacy: Misconceptions and Realities
   Text: The term financial literacy is both appealing and straightforward in its wording. At first glance, its meaning seems obvious: it is the combin

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تثبیت قرارداد Citation کتاب‌ها برای مرحله Ingestion

آزمایش Prototype با موفقیت انجام شد:

`BOOK PARAGRAPH LOCATOR PROTOTYPE GATE: PASS`

در مجموع برای پنج Edition مورد بررسی:

- `3563` Paragraph معنی‌دار شناسایی شد.
- دقیقاً `3563` Locator ساخته شد.
- تمام `3563` شناسه Paragraph یکتا بودند.

همچنین مشخص شد که Duplicate Text در چند کتاب قابل‌توجه است؛ برای مثال در فایل `[4]` تعداد `242` Paragraph عضو گروه‌های متن تکراری هستند.

بنابراین `text_sha256` نمی‌تواند به‌تنهایی شناسه Citation باشد.

از طرف دیگر Headingها نیز در همه کتاب‌ها پوشش یکسانی ندارند. در نتیجه Heading برای Context مفید است، اما Locator اصلی نیست.

قرارداد Citation مورد قبول برای Book Ingestion به این صورت تعریف می‌شود:

`Work ID → Edition ID → Stable Paragraph ID`

که در آن:

`Stable Paragraph ID = Edition ID + Meaningful Paragraph Sequence`

فیلدهای تکمیلی هر Locator:

- `work_id`
- `edition_id`
- `paragraph_id`
- `paragraph_number`
- `text_sha256`
- `heading_path`
- `style`
- `is_heading`
- `in_table`

قاعده مهم:

`text_sha256 = Validation Evidence`

و نه:

`text_sha256 = Paragraph Identity`

شماره صفحه Word نیز به‌عنوان Locator پایدار استفاده نخواهد شد.

این قرارداد مربوط به Snapshot مشخص هر Edition است. اگر محتوای Edition در آینده تغییر کند، باید Edition جدید یا Version جدید ایجاد شود و Paragraph IDهای نسخه قبلی بازنویسی نشوند.

در این مرحله قرارداد فقط Validate می‌شود و هنوز Artifact نهایی Citation ساخته یا ذخیره نمی‌شود. تولید Locatorهای واقعی در `04_book_ingestion.ipynb` انجام خواهد شد.

**معیار PASS:**

- تمام Editionهای مورد بررسی Work ID معتبر داشته باشند.
- تمام Paragraph IDها یکتا باشند.
- هیچ Paragraph ID صرفاً بر اساس Hash متن ساخته نشده باشد.
- Duplicate Text باعث Collision در Paragraph ID نشود.
- قرارداد Citation برای استفاده در Notebook 04 بدون ابهام قابل تعریف باشد.

</div>

In [12]:
# Code Cell 12

work_id_by_edition = {
    edition["edition_id"]: edition["work_id"]
    for edition in edition_records
}

citation_contract_records = []

for file_index, locators in citation_locator_prototypes.items():
    for locator in locators:
        citation_contract_records.append({
            "work_id": work_id_by_edition[locator["edition_id"]],
            "edition_id": locator["edition_id"],
            "paragraph_id": locator["paragraph_id"],
            "paragraph_number": locator["paragraph_number"],
            "text_sha256": locator["text_sha256"],
            "heading_path": locator["heading_path"],
            "style": locator["style"],
            "is_heading": locator["is_heading"],
            "in_table": locator["in_table"],
        })

paragraph_ids = [
    record["paragraph_id"]
    for record in citation_contract_records
]

text_hash_counts = Counter(
    record["text_sha256"]
    for record in citation_contract_records
)

duplicate_hash_groups = sum(
    1 for count in text_hash_counts.values()
    if count > 1
)

duplicate_hash_members = sum(
    count for count in text_hash_counts.values()
    if count > 1
)

all_editions_resolved = all(
    record["edition_id"] in work_id_by_edition
    for record in citation_contract_records
)

all_work_ids_resolved = all(
    record["work_id"] in {
        work["work_id"]
        for work in work_records
    }
    for record in citation_contract_records
)

unique_paragraph_ids = (
    len(paragraph_ids) == len(set(paragraph_ids))
)

citation_contract_pass = (
    len(citation_contract_records) == 3563
    and unique_paragraph_ids
    and all_editions_resolved
    and all_work_ids_resolved
    and duplicate_hash_groups > 0
)

print("BOOK CITATION CONTRACT")
print("-" * 80)
print("Locator identity:")
print("  Work ID → Edition ID → Paragraph ID")
print()
print("Paragraph ID policy:")
print("  <edition_id>:p<meaningful-paragraph-sequence>")
print()
print("Text SHA-256 role:")
print("  CONTENT VALIDATION ONLY")
print()
print("Heading path role:")
print("  STRUCTURAL CONTEXT")
print()
print("DOCX page number role:")
print("  NOT USED AS STABLE LOCATOR")
print()
print("Total prototype locators:", len(citation_contract_records))
print("Unique paragraph IDs:", len(set(paragraph_ids)))
print("Duplicate text-hash groups:", duplicate_hash_groups)
print("Duplicate text-hash member paragraphs:", duplicate_hash_members)
print("All editions resolved:", all_editions_resolved)
print("All Work IDs resolved:", all_work_ids_resolved)
print()
print(
    "BOOK CITATION CONTRACT GATE:",
    "PASS" if citation_contract_pass else "FAIL",
)

BOOK CITATION CONTRACT
--------------------------------------------------------------------------------
Locator identity:
  Work ID → Edition ID → Paragraph ID

Paragraph ID policy:
  <edition_id>:p<meaningful-paragraph-sequence>

Text SHA-256 role:
  CONTENT VALIDATION ONLY

Heading path role:
  STRUCTURAL CONTEXT

DOCX page number role:
  NOT USED AS STABLE LOCATOR

Total prototype locators: 3563
Unique paragraph IDs: 3563
Duplicate text-hash groups: 123
Duplicate text-hash member paragraphs: 615
All editions resolved: True
All Work IDs resolved: True

BOOK CITATION CONTRACT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی Manifestها و Artifactهای قبلی مرتبط با کتاب

قرارداد Citation کتاب‌ها با موفقیت تأیید شد:

`BOOK CITATION CONTRACT GATE: PASS`

قبل از ساخت Book Inventory رسمی باید بررسی شود آیا در پروژه از قبل Artifact یا Manifest مرتبط با کتاب وجود دارد یا خیر.

این مرحله فقط `Discovery` انجام می‌دهد و هیچ فایلی ساخته، حذف یا تغییر داده نمی‌شود.

مسیر اصلی Manifestهای Phase 2:

`data/phase2/manifests/`

همچنین نام فایل‌های موجود در `data/phase2/` و `reports/phase2/` از نظر وجود Artifactهای مرتبط با `book` بررسی می‌شود.

هدف این است که مشخص شود:

- آیا Book Manifest قبلی وجود دارد؟
- آیا Inventory یا Rights Artifact قبلی وجود دارد؟
- آیا نسخه‌ای وجود دارد که باید ابتدا بررسی و Version آن حفظ شود؟
- یا اینکه Notebook 03 باید اولین Book Manifest رسمی پروژه را ایجاد کند؟

قاعده این مرحله:

`DISCOVER → VALIDATE → THEN CREATE`

هیچ Artifact جدیدی تا مشخص شدن وضعیت موجود ساخته نمی‌شود.

**معیار PASS:**

تمام فایل‌های مرتبط با `book` در مسیرهای Phase 2 شناسایی شوند و وضعیت وجود یا عدم وجود Book Manifest قبلی بدون حدس مشخص شود.

</div>

In [13]:
# Code Cell 13

search_roots = [
    project_root / "data" / "phase2" / "manifests",
    project_root / "data" / "phase2",
    project_root / "reports" / "phase2",
]

book_related_files = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        relative_path = path.relative_to(project_root)
        searchable_name = str(relative_path).casefold()

        if any(
            keyword in searchable_name
            for keyword in ("book", "books", "کتاب")
        ):
            book_related_files.append(path)

book_related_files = sorted(
    set(book_related_files),
    key=lambda path: str(path.relative_to(project_root)).casefold(),
)

manifest_dir = project_root / "data" / "phase2" / "manifests"

manifest_files = (
    sorted(
        [path for path in manifest_dir.iterdir() if path.is_file()],
        key=lambda path: path.name.casefold(),
    )
    if manifest_dir.exists()
    else []
)

book_manifest_candidates = [
    path
    for path in manifest_files
    if "book" in path.name.casefold()
]

print("PHASE 2 MANIFEST DIRECTORY")
print("-" * 90)
print("Path:", manifest_dir.relative_to(project_root))
print("Exists:", manifest_dir.exists())
print("Manifest files found:", len(manifest_files))

for path in manifest_files:
    print(" -", path.name)

print()
print("BOOK-RELATED FILES DISCOVERED")
print("-" * 90)

if book_related_files:
    for path in book_related_files:
        print(" -", path.relative_to(project_root))
else:
    print(" - NONE")

print()
print("Book manifest candidates:", len(book_manifest_candidates))

for path in book_manifest_candidates:
    print(" -", path.relative_to(project_root))

print()
print(
    "Existing Book Manifest:",
    "YES" if book_manifest_candidates else "NO",
)

discovery_gate = manifest_dir.exists()

print()
print(
    "BOOK ARTIFACT DISCOVERY GATE:",
    "PASS" if discovery_gate else "FAIL",
)

PHASE 2 MANIFEST DIRECTORY
------------------------------------------------------------------------------------------
Path: data/phase2/manifests
Exists: True
Manifest files found: 2
 - book_manifest_v1.json
 - website_manifest_v1.json

BOOK-RELATED FILES DISCOVERED
------------------------------------------------------------------------------------------
 - data/phase2/manifests/book_manifest_v1.json
 - reports/phase2/book_inventory_acceptance_v1.json

Book manifest candidates: 1
 - data/phase2/manifests/book_manifest_v1.json

Existing Book Manifest: YES

BOOK ARTIFACT DISCOVERY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Idempotent اولین Book Manifest رسمی پروژه

در اجرای اولیه، `book_manifest_v1.json` با موفقیت ساخته و سپس مستقل Validate شد:

`BOOK MANIFEST INDEPENDENT VALIDATION GATE: PASS`

با این حال یک مشکل Reproducibility در منطق اولیه این Cell شناسایی شد.

نسخه قبلی در صورت وجود `book_manifest_v1.json` اجرای Notebook را با `FileExistsError` متوقف می‌کرد. این رفتار در توسعه از overwrite ناخواسته جلوگیری می‌کرد، اما با Gate نهایی پروژه ناسازگار بود:

`Restart Kernel → Run All → Zero Error`

بنابراین منطق این Cell با حداقل تغییر اصلاح می‌شود:

- Manifest مورد انتظار دوباره به‌صورت deterministic ساخته می‌شود.
- اگر فایل هنوز وجود نداشته باشد، ایجاد می‌شود.
- اگر فایل وجود داشته و محتوای آن دقیقاً با Manifest مورد انتظار یکسان باشد، بدون overwrite مجدداً پذیرفته می‌شود.
- اگر فایل موجود با Manifest مورد انتظار متفاوت باشد، اجرا Fail می‌شود و فایل موجود به‌صورت Silent تغییر نمی‌کند.

قاعده:

`Existing + Identical → REUSED_IDENTICAL`

`Existing + Different → FAIL`

این اصلاح هیچ تغییری در Contract، Workها، Editionها، Hashها یا تصمیم‌های Inventory ایجاد نمی‌کند و فقط قابلیت اجرای مجدد Notebook را تضمین می‌کند.

**معیار PASS:**

Manifest یا برای اولین بار ایجاد شود یا نسخه موجود دقیقاً با Candidate فعلی یکسان باشد.

</div>

In [14]:
# Code Cell 14

import hashlib
import json

manifest_path = project_root / "data" / "phase2" / "manifests" / "book_manifest_v1.json"

attribution_by_work = {
    record["work_id"]: record
    for record in work_attribution_records
}

decision_by_file = {
    record["file_index"]: record
    for record in ingestion_decisions
}

edition_by_file = {
    record["file_index"]: record
    for record in edition_records
}

inventory_by_file = {
    index + 1: item
    for index, item in enumerate(book_inventory)
}

manifest_works = []

for work in work_records:
    attribution = attribution_by_work[work["work_id"]]

    manifest_works.append({
        "work_id": work["work_id"],
        "title": work["title"],
        "author": attribution["author"],
        "knowledge_owner": attribution["knowledge_owner"],
        "attribution_basis": attribution["attribution_basis"],
        "source_edition_id": work["source_edition_id"],
        "rights_holder": None,
        "rights_status": attribution["rights_status"],
        "rights_evidence_id": None,
        "allowed_uses": None,
    })

manifest_editions = []

for edition in edition_records:
    file_index = edition["file_index"]
    decision = decision_by_file[file_index]

    manifest_editions.append({
        "edition_id": edition["edition_id"],
        "work_id": edition["work_id"],
        "language": edition["language"],
        "edition_role": edition["edition_role"],
        "file_index": file_index,
        "ingestion_status": decision["decision"],
        "ingestion_reason": decision["reason"],
    })

manifest_files = []

for file_index, path in enumerate(book_files, start=1):
    inventory_item = inventory_by_file[file_index]
    edition = edition_by_file[file_index]

    manifest_files.append({
        "file_index": file_index,
        "edition_id": edition["edition_id"],
        "relative_path": str(path.relative_to(project_root)),
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "file_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "text_sha256": inventory_item["text_sha256"],
        "meaningful_paragraphs": inventory_item["paragraph_count"],
    })

book_manifest = {
    "manifest_type": "phase2_book_inventory",
    "manifest_version": "v1",
    "source_type": "book",
    "work_count": len(manifest_works),
    "edition_count": len(manifest_editions),
    "physical_file_count": len(manifest_files),
    "citation_contract": {
        "identity": "work_id -> edition_id -> paragraph_id",
        "paragraph_id_policy": "<edition_id>:p<meaningful-paragraph-sequence>",
        "text_sha256_role": "content_validation_only",
        "heading_path_role": "structural_context",
        "docx_page_number_role": "not_used_as_stable_locator",
    },
    "works": manifest_works,
    "editions": manifest_editions,
    "physical_files": manifest_files,
}

manifest_path.parent.mkdir(parents=True, exist_ok=True)

if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as file:
        existing_manifest = json.load(file)

    if existing_manifest != book_manifest:
        raise ValueError(
            "Existing book_manifest_v1.json differs from the deterministic candidate. "
            "Refusing to overwrite."
        )

    manifest_action = "REUSED_IDENTICAL"

else:
    with manifest_path.open("w", encoding="utf-8") as file:
        json.dump(
            book_manifest,
            file,
            ensure_ascii=False,
            indent=2,
        )

    manifest_action = "CREATED"

creation_pass = (
    manifest_path.exists()
    and len(manifest_works) == 4
    and len(manifest_editions) == 7
    and len(manifest_files) == 7
    and manifest_action in {"CREATED", "REUSED_IDENTICAL"}
)

print("BOOK MANIFEST")
print("-" * 80)
print("Path:", manifest_path.relative_to(project_root))
print("Action:", manifest_action)
print("Works:", len(manifest_works))
print("Editions:", len(manifest_editions))
print("Physical files:", len(manifest_files))
print()
print(
    "BOOK MANIFEST CREATION GATE:",
    "PASS" if creation_pass else "FAIL",
)

BOOK MANIFEST
--------------------------------------------------------------------------------
Path: data/phase2/manifests/book_manifest_v1.json
Action: REUSED_IDENTICAL
Works: 4
Editions: 7
Physical files: 7

BOOK MANIFEST CREATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### Independent Re-validation فایل `book_manifest_v1.json`

ساخت اولین Book Manifest رسمی پروژه با موفقیت انجام شد:

`BOOK MANIFEST CREATION GATE: PASS`

اما صرف ساخته‌شدن فایل برای پذیرش آن کافی نیست.

در این مرحله فایل:

`data/phase2/manifests/book_manifest_v1.json`

مستقیماً از Disk دوباره خوانده می‌شود و بدون اتکا به Object ساخته‌شده در Cell قبلی بررسی خواهد شد.

Validation شامل موارد زیر است:

- JSON معتبر باشد.
- `manifest_version = v1`
- دقیقاً `4` Work وجود داشته باشد.
- دقیقاً `7` Edition وجود داشته باشد.
- دقیقاً `7` Physical File وجود داشته باشد.
- تمام `work_id`ها یکتا باشند.
- تمام `edition_id`ها یکتا باشند.
- هر Edition به Work موجود متصل باشد.
- هر Physical File به Edition موجود متصل باشد.
- تمام فایل‌های ثبت‌شده واقعاً روی Disk وجود داشته باشند.
- اندازه فایل و `SHA-256` ثبت‌شده با فایل واقعی برابر باشد.
- Source Edition هر Work واقعاً در Editions وجود داشته باشد.
- وضعیت Rights همچنان `EVIDENCE_REQUIRED` باشد.
- تصمیم‌های اصلی Ingestion بدون تغییر حفظ شده باشند.

این Validation مستقل تعیین می‌کند که Manifest می‌تواند به‌عنوان Artifact معتبر Notebook 03 پذیرفته شود یا خیر.

**معیار PASS:**

تمام Contractهای بالا بدون mismatch تأیید شوند.

</div>

In [15]:
# Code Cell 15

import hashlib
import json

with manifest_path.open("r", encoding="utf-8") as file:
    disk_manifest = json.load(file)

disk_works = disk_manifest.get("works", [])
disk_editions = disk_manifest.get("editions", [])
disk_files = disk_manifest.get("physical_files", [])

disk_work_ids = [item.get("work_id") for item in disk_works]
disk_edition_ids = [item.get("edition_id") for item in disk_editions]

work_id_set = set(disk_work_ids)
edition_id_set = set(disk_edition_ids)

edition_work_mismatches = [
    item.get("edition_id")
    for item in disk_editions
    if item.get("work_id") not in work_id_set
]

file_edition_mismatches = [
    item.get("filename")
    for item in disk_files
    if item.get("edition_id") not in edition_id_set
]

source_edition_mismatches = [
    item.get("work_id")
    for item in disk_works
    if item.get("source_edition_id") not in edition_id_set
]

missing_physical_files = []
size_mismatches = []
hash_mismatches = []

for item in disk_files:
    physical_path = project_root / item["relative_path"]

    if not physical_path.exists():
        missing_physical_files.append(item["relative_path"])
        continue

    actual_size = physical_path.stat().st_size
    actual_hash = hashlib.sha256(
        physical_path.read_bytes()
    ).hexdigest()

    if actual_size != item["size_bytes"]:
        size_mismatches.append(item["relative_path"])

    if actual_hash != item["file_sha256"]:
        hash_mismatches.append(item["relative_path"])

rights_statuses = {
    item.get("rights_status")
    for item in disk_works
}

decision_by_index = {
    item["file_index"]: item["ingestion_status"]
    for item in disk_editions
}

expected_decisions = {
    1: "AUTHORITATIVE_REFERENCE",
    2: "QUARANTINED_PENDING_TRANSLATION_QA",
    3: "ACCEPTED_FOR_INGESTION",
    4: "ACCEPTED_FOR_INGESTION",
    5: "NON_CANONICAL",
    6: "ACCEPTED_FOR_INGESTION",
    7: "NON_CANONICAL",
}

decision_mismatches = {
    index: {
        "expected": expected,
        "actual": decision_by_index.get(index),
    }
    for index, expected in expected_decisions.items()
    if decision_by_index.get(index) != expected
}

independent_validation_pass = (
    disk_manifest.get("manifest_version") == "v1"
    and disk_manifest.get("manifest_type") == "phase2_book_inventory"
    and len(disk_works) == 4
    and len(disk_work_ids) == len(set(disk_work_ids)) == 4
    and len(disk_editions) == 7
    and len(disk_edition_ids) == len(set(disk_edition_ids)) == 7
    and len(disk_files) == 7
    and not edition_work_mismatches
    and not file_edition_mismatches
    and not source_edition_mismatches
    and not missing_physical_files
    and not size_mismatches
    and not hash_mismatches
    and rights_statuses == {"EVIDENCE_REQUIRED"}
    and not decision_mismatches
)

print("BOOK MANIFEST INDEPENDENT VALIDATION")
print("-" * 80)
print("Manifest type:", disk_manifest.get("manifest_type"))
print("Manifest version:", disk_manifest.get("manifest_version"))
print("Works:", len(disk_works))
print("Editions:", len(disk_editions))
print("Physical files:", len(disk_files))
print("Duplicate Work IDs:", len(disk_work_ids) - len(set(disk_work_ids)))
print("Duplicate Edition IDs:", len(disk_edition_ids) - len(set(disk_edition_ids)))
print("Edition → Work mismatches:", len(edition_work_mismatches))
print("File → Edition mismatches:", len(file_edition_mismatches))
print("Source edition mismatches:", len(source_edition_mismatches))
print("Missing physical files:", len(missing_physical_files))
print("Size mismatches:", len(size_mismatches))
print("File hash mismatches:", len(hash_mismatches))
print("Rights statuses:", sorted(rights_statuses))
print("Ingestion decision mismatches:", len(decision_mismatches))
print()
print(
    "BOOK MANIFEST INDEPENDENT VALIDATION GATE:",
    "PASS" if independent_validation_pass else "FAIL",
)

BOOK MANIFEST INDEPENDENT VALIDATION
--------------------------------------------------------------------------------
Manifest type: phase2_book_inventory
Manifest version: v1
Works: 4
Editions: 7
Physical files: 7
Duplicate Work IDs: 0
Duplicate Edition IDs: 0
Edition → Work mismatches: 0
File → Edition mismatches: 0
Source edition mismatches: 0
Missing physical files: 0
Size mismatches: 0
File hash mismatches: 0
Rights statuses: ['EVIDENCE_REQUIRED']
Ingestion decision mismatches: 0

BOOK MANIFEST INDEPENDENT VALIDATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### جست‌وجوی شواهد حقوقی موجود در Repository

تا این مرحله Attribution و هویت چهار Work مشخص شده است، اما وضعیت حقوقی آن‌ها عمداً به‌صورت:

`EVIDENCE_REQUIRED`

ثبت شده است.

قبل از پذیرش نهایی Notebook 03 باید بررسی شود آیا در Repository فایل یا سندی وجود دارد که بتواند به‌عنوان Evidence حقوقی استفاده شود؛ برای مثال فایل‌هایی مرتبط با:

- `rights`
- `license`
- `permission`
- `consent`
- `copyright`
- `authorization`
- `مجوز`
- `رضایت`
- `حق نشر`
- `حقوق`

این مرحله فقط نام و مسیر فایل‌های Candidate را جست‌وجو می‌کند و هیچ نتیجه حقوقی را از روی نام فایل حدس نمی‌زند.

اگر هیچ Candidateای پیدا نشود، وضعیت:

`EVIDENCE_REQUIRED`

به‌درستی حفظ می‌شود.

اگر Candidate پیدا شود، در مرحله بعد باید محتوای همان فایل به‌صورت مستقل بررسی شود و صرف نام آن Evidence محسوب نمی‌شود.

**معیار PASS:**

جست‌وجوی Repository بدون تغییر فایل‌ها انجام شود و وضعیت وجود یا عدم وجود Candidateهای حقوقی مشخص شود.

</div>

In [16]:
# Code Cell 16

rights_keywords = (
    "rights",
    "license",
    "licence",
    "permission",
    "consent",
    "copyright",
    "authorization",
    "authorisation",
    "مجوز",
    "رضایت",
    "حق نشر",
    "حقوق",
)

excluded_roots = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
}

rights_candidates = []

for path in project_root.rglob("*"):
    if not path.is_file():
        continue

    relative = path.relative_to(project_root)

    if any(part in excluded_roots for part in relative.parts):
        continue

    searchable = str(relative).casefold()

    if any(keyword.casefold() in searchable for keyword in rights_keywords):
        rights_candidates.append(relative)

rights_candidates = sorted(
    set(rights_candidates),
    key=lambda path: str(path).casefold(),
)

print("BOOK RIGHTS EVIDENCE DISCOVERY")
print("-" * 80)
print("Candidate files found:", len(rights_candidates))

if rights_candidates:
    for path in rights_candidates:
        print(" -", path)
else:
    print(" - NONE")

print()
print(
    "Current rights status:",
    "REVIEW_REQUIRED" if rights_candidates else "EVIDENCE_REQUIRED",
)
print()
print("BOOK RIGHTS EVIDENCE DISCOVERY GATE: PASS")

BOOK RIGHTS EVIDENCE DISCOVERY
--------------------------------------------------------------------------------
Candidate files found: 0
 - NONE

Current rights status: EVIDENCE_REQUIRED

BOOK RIGHTS EVIDENCE DISCOVERY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت گزارش پذیرش Notebook 03

جست‌وجوی Evidence حقوقی با موفقیت انجام شد:

`BOOK RIGHTS EVIDENCE DISCOVERY GATE: PASS`

هیچ فایل Candidate برای Rights در Repository پیدا نشد؛ بنابراین وضعیت حقوقی چهار Work همچنان:

`EVIDENCE_REQUIRED`

باقی می‌ماند.

در این مرحله نتایج اندازه‌گیری‌شده Notebook 03 در یک گزارش مستقل ثبت می‌شوند:

`reports/phase2/book_inventory_acceptance_v1.json`

این گزارش نتیجه Inventory را از آمادگی Ingestion تفکیک می‌کند.

وضعیت فعلی:

- `4` Work مستقل شناسایی شده است.
- `7` Edition / Physical File ثبت شده است.
- فایل‌های `[3]`، `[4]` و `[6]` Sourceهای فارسی اصلی هستند.
- فایل `[1]` مرجع اصلی انگلیسی `Money and Me` است.
- فایل `[2]` ترجمه فارسی است و تا QA مستقل ترجمه در Quarantine باقی می‌ماند.
- فایل‌های `[5]` و `[7]` نسخه‌های غیر Canonical هستند.
- قرارداد Citation بر پایه `Work ID → Edition ID → Paragraph ID` تأیید شده است.
- `DOCX page number` Locator پایدار محسوب نمی‌شود.
- Rights Evidence هنوز وجود ندارد.

بنابراین Notebook 03 می‌تواند از نظر Inventory و Identity موفق باشد، در حالی که محدودیت‌های باز برای مراحل بعد صریحاً حفظ می‌شوند.

این Cell به‌صورت Idempotent طراحی می‌شود: اگر گزارش موجود دقیقاً یکسان باشد دوباره استفاده می‌شود و اگر متفاوت باشد بدون بررسی overwrite نخواهد شد.

**معیار PASS:**

گزارش دقیقاً وضعیت اندازه‌گیری‌شده Notebook 03 را ثبت کند و هیچ محدودیت باز یا تصمیم Quarantine مخفی نشود.

</div>

In [17]:
# Code Cell 17

import json

acceptance_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "book_inventory_acceptance_v1.json"
)

book_inventory_report = {
    "report_type": "phase2_book_inventory_acceptance",
    "report_version": "v1",
    "notebook": "03_book_inventory.ipynb",
    "inventory_status": "PASS",
    "physical_file_count": 7,
    "work_count": 4,
    "edition_count": 7,
    "author": "Dr. Komeil Roudi",
    "knowledge_owner": "Dr. Komeil Roudi",
    "accepted_for_ingestion_file_indices": [3, 4, 6],
    "authoritative_reference_file_indices": [1],
    "quarantined_file_indices": [2],
    "non_canonical_file_indices": [5, 7],
    "open_gates": {
        "translation_qa": {
            "status": "REQUIRED",
            "file_index": 2,
            "edition_id": "book:money-and-me:fa-user-translation",
        },
        "rights_evidence": {
            "status": "EVIDENCE_REQUIRED",
            "repository_candidate_files_found": 0,
        },
    },
    "citation_contract": {
        "identity": "work_id -> edition_id -> paragraph_id",
        "paragraph_id_policy": "<edition_id>:p<meaningful-paragraph-sequence>",
        "prototype_locator_count": 3563,
        "unique_paragraph_id_count": 3563,
        "duplicate_text_hash_groups": 123,
        "duplicate_text_hash_member_paragraphs": 615,
        "text_sha256_role": "content_validation_only",
        "heading_path_role": "structural_context",
        "docx_page_number_role": "not_used_as_stable_locator",
    },
    "next_notebook": "04_book_ingestion.ipynb",
    "constraints_for_next_notebook": [
        "Do not ingest file 2 until translation QA is accepted.",
        "Do not ingest files 5 or 7 as canonical sources.",
        "Preserve Work ID, Edition ID, file hashes, and source lineage.",
        "Preserve rights_status as EVIDENCE_REQUIRED until evidence exists.",
        "Do not use DOCX page numbers as stable citation locators.",
        "Do not perform new chunking in Notebook 04.",
    ],
}

acceptance_report_path.parent.mkdir(parents=True, exist_ok=True)

if acceptance_report_path.exists():
    with acceptance_report_path.open("r", encoding="utf-8") as file:
        existing_report = json.load(file)

    if existing_report != book_inventory_report:
        raise ValueError(
            "Existing book_inventory_acceptance_v1.json differs from the "
            "deterministic candidate. Refusing to overwrite."
        )

    report_action = "REUSED_IDENTICAL"

else:
    with acceptance_report_path.open("w", encoding="utf-8") as file:
        json.dump(
            book_inventory_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

    report_action = "CREATED"

report_creation_pass = (
    acceptance_report_path.exists()
    and book_inventory_report["inventory_status"] == "PASS"
    and book_inventory_report["physical_file_count"] == 7
    and book_inventory_report["work_count"] == 4
    and book_inventory_report["edition_count"] == 7
    and book_inventory_report["quarantined_file_indices"] == [2]
    and book_inventory_report["non_canonical_file_indices"] == [5, 7]
    and book_inventory_report["open_gates"]["rights_evidence"]["status"]
    == "EVIDENCE_REQUIRED"
)

print("BOOK INVENTORY ACCEPTANCE REPORT")
print("-" * 80)
print("Path:", acceptance_report_path.relative_to(project_root))
print("Action:", report_action)
print("Inventory status:", book_inventory_report["inventory_status"])
print("Works:", book_inventory_report["work_count"])
print("Editions:", book_inventory_report["edition_count"])
print(
    "Accepted for ingestion:",
    book_inventory_report["accepted_for_ingestion_file_indices"],
)
print("Quarantined:", book_inventory_report["quarantined_file_indices"])
print("Non-canonical:", book_inventory_report["non_canonical_file_indices"])
print(
    "Rights:",
    book_inventory_report["open_gates"]["rights_evidence"]["status"],
)
print(
    "Translation QA:",
    book_inventory_report["open_gates"]["translation_qa"]["status"],
)
print()
print(
    "BOOK INVENTORY ACCEPTANCE REPORT GATE:",
    "PASS" if report_creation_pass else "FAIL",
)

BOOK INVENTORY ACCEPTANCE REPORT
--------------------------------------------------------------------------------
Path: reports/phase2/book_inventory_acceptance_v1.json
Action: REUSED_IDENTICAL
Inventory status: PASS
Works: 4
Editions: 7
Accepted for ingestion: [3, 4, 6]
Quarantined: [2]
Non-canonical: [5, 7]
Rights: EVIDENCE_REQUIRED
Translation QA: REQUIRED

BOOK INVENTORY ACCEPTANCE REPORT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### Independent Re-validation گزارش پذیرش Book Inventory

گزارش:

`reports/phase2/book_inventory_acceptance_v1.json`

با موفقیت ایجاد شد:

`BOOK INVENTORY ACCEPTANCE REPORT GATE: PASS`

در این مرحله گزارش ذخیره‌شده مستقیماً از Disk دوباره خوانده می‌شود تا مستقل از Variableهای Cell قبلی Validation شود.

موارد بررسی‌شده شامل:

- نوع و Version گزارش
- وضعیت نهایی Inventory
- تعداد `Work`، `Edition` و Physical File
- فایل‌های پذیرفته‌شده
- فایل Quarantine
- نسخه‌های `NON_CANONICAL`
- وضعیت باز Translation QA
- وضعیت `Rights Evidence`
- قرارداد Citation
- محدودیت‌های ثبت‌شده

این مرحله هیچ Artifact جدیدی ایجاد یا اصلاح نمی‌کند.

**معیار PASS:**

تمام مقادیر کلیدی گزارش ذخیره‌شده با نتایج واقعی Notebook 03 سازگار باشند و هیچ محدودیت باز حذف یا تغییر نکرده باشد.

</div>

In [18]:
# Code Cell 18

import json

with acceptance_report_path.open("r", encoding="utf-8") as file:
    disk_report = json.load(file)

validation_checks = {
    "report_type": (
        disk_report.get("report_type")
        == "phase2_book_inventory_acceptance"
    ),
    "report_version": disk_report.get("report_version") == "v1",
    "inventory_status": disk_report.get("inventory_status") == "PASS",
    "physical_file_count": disk_report.get("physical_file_count") == 7,
    "work_count": disk_report.get("work_count") == 4,
    "edition_count": disk_report.get("edition_count") == 7,
    "accepted_files": (
        disk_report.get("accepted_for_ingestion_file_indices")
        == [3, 4, 6]
    ),
    "authoritative_reference": (
        disk_report.get("authoritative_reference_file_indices")
        == [1]
    ),
    "quarantined_files": (
        disk_report.get("quarantined_file_indices")
        == [2]
    ),
    "non_canonical_files": (
        disk_report.get("non_canonical_file_indices")
        == [5, 7]
    ),
    "translation_qa": (
        disk_report.get("open_gates", {})
        .get("translation_qa", {})
        .get("status")
        == "REQUIRED"
    ),
    "rights_evidence": (
        disk_report.get("open_gates", {})
        .get("rights_evidence", {})
        .get("status")
        == "EVIDENCE_REQUIRED"
    ),
    "citation_locator_count": (
        disk_report.get("citation_contract", {})
        .get("prototype_locator_count")
        == 3563
    ),
    "citation_unique_ids": (
        disk_report.get("citation_contract", {})
        .get("unique_paragraph_id_count")
        == 3563
    ),
    "docx_page_policy": (
        disk_report.get("citation_contract", {})
        .get("docx_page_number_role")
        == "not_used_as_stable_locator"
    ),
}

failed_checks = [
    name
    for name, passed in validation_checks.items()
    if not passed
]

print("BOOK INVENTORY REPORT INDEPENDENT VALIDATION")
print("-" * 80)

for name, passed in validation_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print()
print("Checks:", len(validation_checks))
print("Failed checks:", len(failed_checks))

if failed_checks:
    print("Failures:")
    for name in failed_checks:
        print(" -", name)

print()
print(
    "BOOK INVENTORY REPORT INDEPENDENT VALIDATION GATE:",
    "PASS" if not failed_checks else "FAIL",
)

BOOK INVENTORY REPORT INDEPENDENT VALIDATION
--------------------------------------------------------------------------------
report_type: PASS
report_version: PASS
inventory_status: PASS
physical_file_count: PASS
work_count: PASS
edition_count: PASS
accepted_files: PASS
authoritative_reference: PASS
quarantined_files: PASS
non_canonical_files: PASS
translation_qa: PASS
rights_evidence: PASS
citation_locator_count: PASS
citation_unique_ids: PASS
docx_page_policy: PASS

Checks: 15
Failed checks: 0

BOOK INVENTORY REPORT INDEPENDENT VALIDATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### Final Acceptance — Book Inventory

تمام بخش‌های اصلی `03_book_inventory.ipynb` تا این نقطه با موفقیت Validation شده‌اند.

نتایج نهایی Inventory:

- `7` فایل فیزیکی `DOCX` شناسایی شد.
- این فایل‌ها به `4` Work مستقل تعلق دارند.
- `7` Edition ثبت شد.
- فایل‌های `[3]`، `[4]` و `[6]` منابع اصلی فارسی مولف و `ACCEPTED_FOR_INGESTION` هستند.
- فایل `[1]` نسخه انگلیسی اصلی `Money and Me` و `AUTHORITATIVE_REFERENCE` است.
- فایل `[2]` ترجمه فارسی همان Work است و تا انجام QA مستقل ترجمه در وضعیت `QUARANTINED_PENDING_TRANSLATION_QA` باقی می‌ماند.
- فایل‌های `[5]` و `[7]` نسخه‌های بازسازی‌شده یا بازطراحی‌شده و `NON_CANONICAL` هستند.
- مولف و `knowledge_owner` چهار Work، دکتر کمیل رودی ثبت شده است.
- Evidence حقوقی مستقلی در Repository پیدا نشد و `rights_status = EVIDENCE_REQUIRED` باقی می‌ماند.
- قرارداد Citation کتاب‌ها بر پایه:

`Work ID → Edition ID → Stable Paragraph ID`

تأیید شده است.

- `Heading Path` فقط Context ساختاری است.
- `text_sha256` فقط برای Validation محتوا استفاده می‌شود.
- شماره صفحه `DOCX` به‌عنوان Locator پایدار استفاده نمی‌شود.
- `3563` Paragraph Locator آزمایشی با `3563` شناسه یکتا Validation شده است.
- Book Manifest و Acceptance Report هر دو از روی Disk به‌صورت مستقل Validation شده‌اند.

Artifactهای اصلی Notebook:

`data/phase2/manifests/book_manifest_v1.json`

`reports/phase2/book_inventory_acceptance_v1.json`

موارد باز، Failure این Notebook محسوب نمی‌شوند زیرا خارج از Scope مرحله Inventory هستند:

- QA ترجمه فارسی `Money and Me`
- Rights Evidence

این Notebook هیچ Book Ingestion و هیچ Chunking انجام نداده است.

**Final Acceptance فقط زمانی PASS است که Artifactهای اصلی موجود باشند، Validationهای مستقل موفق باشند و محدودیت‌های باز بدون پنهان‌سازی در گزارش ثبت شده باشند.**

</div>

In [19]:
# Code Cell 19

final_acceptance_checks = {
    "book_file_discovery": len(book_files) == 7,
    "work_count": len(work_records) == 4,
    "edition_count": len(edition_records) == 7,
    "work_model": model_pass,
    "attribution_model": attribution_model_pass,
    "ingestion_selection": decision_gate,
    "citation_prototype": citation_model_pass,
    "citation_contract": citation_contract_pass,
    "manifest_exists": manifest_path.exists(),
    "manifest_independent_validation": independent_validation_pass,
    "acceptance_report_exists": acceptance_report_path.exists(),
    "acceptance_report_independent_validation": not failed_checks,
    "translation_qa_preserved": (
        disk_report["open_gates"]["translation_qa"]["status"] == "REQUIRED"
    ),
    "rights_evidence_preserved": (
        disk_report["open_gates"]["rights_evidence"]["status"]
        == "EVIDENCE_REQUIRED"
    ),
}

final_failed_checks = [
    name
    for name, passed in final_acceptance_checks.items()
    if not passed
]

print("NOTEBOOK 03 FINAL ACCEPTANCE")
print("-" * 80)

for name, passed in final_acceptance_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print()
print("Final checks:", len(final_acceptance_checks))
print("Failed checks:", len(final_failed_checks))
print("Works accepted:", 4)
print("Physical files inventoried:", 7)
print("Files accepted for ingestion:", [3, 4, 6])
print("Authoritative reference:", [1])
print("Translation QA quarantine:", [2])
print("Non-canonical files:", [5, 7])
print("Rights status: EVIDENCE_REQUIRED")
print()

print(
    "NOTEBOOK 03 FINAL ACCEPTANCE:",
    "PASS" if not final_failed_checks else "FAIL",
)

NOTEBOOK 03 FINAL ACCEPTANCE
--------------------------------------------------------------------------------
book_file_discovery: PASS
work_count: PASS
edition_count: PASS
work_model: PASS
attribution_model: PASS
ingestion_selection: PASS
citation_prototype: PASS
citation_contract: PASS
manifest_exists: PASS
manifest_independent_validation: PASS
acceptance_report_exists: PASS
acceptance_report_independent_validation: PASS
translation_qa_preserved: PASS
rights_evidence_preserved: PASS

Final checks: 14
Failed checks: 0
Works accepted: 4
Physical files inventoried: 7
Files accepted for ingestion: [3, 4, 6]
Authoritative reference: [1]
Translation QA quarantine: [2]
Non-canonical files: [5, 7]
Rights status: EVIDENCE_REQUIRED

NOTEBOOK 03 FINAL ACCEPTANCE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 03 — Final Summary

هدف `03_book_inventory.ipynb` ایجاد Inventory رسمی، قابل‌ردیابی و قابل‌بازتولید برای منابع کتابی Phase 2 پیش از شروع Book Ingestion بود.

در این Notebook هیچ Book Ingestion، Chunking یا پردازش مربوط به Notebookهای بعدی انجام نشد.

### ورودی واقعی

پوشه:

`data/books/`

شامل `7` فایل فیزیکی `DOCX` بود.

بررسی هویت و شباهت محتوا نشان داد که تعداد فایل‌های فیزیکی برابر با تعداد آثار مستقل نیست.

### مدل نهایی منابع

در مجموع:

- `7` Physical File
- `7` Edition
- `4` Work مستقل

شناسایی و ثبت شد.

#### Work 1 — `Money and Me`

`book:money-and-me`

- فایل `[1]` نسخه انگلیسی اصلی مولف و `AUTHORITATIVE_REFERENCE` است.
- فایل `[2]` ترجمه فارسی تهیه‌شده از نسخه انگلیسی است.
- فایل `[2]` برای RAG فارسی اهمیت دارد اما تا انجام QA مستقل ترجمه:

`QUARANTINED_PENDING_TRANSLATION_QA`

باقی می‌ماند.

#### Work 2 — نوشتار آموزشی دوره آموزش سواد مالی بزرگسالان

`book:adult-financial-literacy-course`

- فایل `[3]` منبع اصلی فارسی مولف است.
- وضعیت:

`ACCEPTED_FOR_INGESTION`

#### Work 3 — سواد مالی برای همه

`book:financial-literacy-for-all`

- فایل `[4]` منبع اصلی فارسی مولف است.
- فایل `[5]` نسخه بازسازی‌شده همان Work است.
- شباهت محتوایی `[4]` و `[5]` بسیار بالا تأیید شد.
- `[4]` برای Ingestion پذیرفته شد.
- `[5]` به‌عنوان `NON_CANONICAL` فقط برای حفظ Lineage نگهداری می‌شود.

#### Work 4 — نوشتار آموزشی تربیت مربی سواد مالی

`book:trainer-training-financial-literacy`

- فایل `[6]` منبع اصلی مولف است.
- فایل `[7]` دارای متن استخراج‌شده کاملاً یکسان با `[6]` است.
- `[6]` برای Ingestion پذیرفته شد.
- `[7]` به‌عنوان `NON_CANONICAL` حفظ می‌شود.

### Attribution

برای هر چهار Work:

`author = Dr. Komeil Roudi`

و:

`knowledge_owner = Dr. Komeil Roudi`

ثبت شد.

Metadata فنی Word مانند `ASUS`، `sajad yazdani` و `faty gh` به‌عنوان نویسنده تلقی نشد.

### Rights

هیچ Rights Evidence مستقلی در Repository پیدا نشد.

بنابراین برای هر چهار Work:

`rights_status = EVIDENCE_REQUIRED`

حفظ شد.

وجود فایل یا تأیید Authorship به‌تنهایی معادل Rights Evidence در نظر گرفته نشد.

### Citation Contract

ساختار داخلی نسخه‌های مرتبط بررسی شد.

در Prototype:

- `3563` Paragraph معنی‌دار
- `3563` Paragraph Locator
- `3563` Paragraph ID یکتا

تأیید شد.

همچنین:

- `123` گروه Hash متنی تکراری
- `615` Paragraph عضو این گروه‌های تکراری

شناسایی شد.

به همین دلیل `text_sha256` به‌تنهایی Paragraph Identity محسوب نمی‌شود.

قرارداد Citation پذیرفته‌شده:

`Work ID → Edition ID → Stable Paragraph ID`

Policy شناسه Paragraph:

`<edition_id>:p<meaningful-paragraph-sequence>`

نقش سایر فیلدها:

- `text_sha256` → Content Validation
- `heading_path` → Structural Context
- `DOCX page number` → Not a stable locator

### Artifactهای ساخته‌شده و Accepted

`data/phase2/manifests/book_manifest_v1.json`

این Manifest به‌صورت مستقل از Disk Validation شد و تمام Workها، Editionها، Physical Fileها، Hashها و روابط آن بدون mismatch تأیید شدند.

`reports/phase2/book_inventory_acceptance_v1.json`

این گزارش نیز از روی Disk مستقل Validation شد.

### موارد باز

دو مورد عمداً باز باقی مانده‌اند:

1. QA مستقل ترجمه فارسی `Money and Me`
2. Rights Evidence

این موارد Failure مرحله Inventory محسوب نمی‌شوند، اما باید در مراحل مناسب بعدی بدون حذف یا پنهان‌سازی پیگیری شوند.

### محدودیت مرحله

Notebook 03 فقط Inventory، Identity، Attribution، Source Selection و Citation Contract را تعیین کرده است.

در این Notebook:

- Book Ingestion انجام نشده است.
- Chunking انجام نشده است.
- فایل `[2]` هنوز برای Corpus نهایی پذیرفته نشده است.
- فایل‌های `[5]` و `[7]` به‌عنوان منبع Canonical پذیرفته نشده‌اند.

### Final Acceptance

تمام `14` بررسی Final Acceptance موفق بودند:

`NOTEBOOK 03 FINAL ACCEPTANCE: PASS`

Gate نهایی Notebook پس از:

`Restart Kernel → Run All → Zero Error → No unexecuted Code Cells → Notebook Health PASS`

تکمیل می‌شود.

</div>

In [20]:
# Code Cell 20

import json
import re
from pathlib import Path

notebook_path = project_root / "notebooks" / "phase2" / "03_book_inventory.ipynb"

if not notebook_path.exists():
    raise FileNotFoundError(
        f"Notebook not found: {notebook_path.relative_to(project_root)}"
    )

with notebook_path.open("r", encoding="utf-8") as file:
    notebook_data = json.load(file)

cells = notebook_data.get("cells", [])
code_cells = [
    cell
    for cell in cells
    if cell.get("cell_type") == "code"
]

execution_counts = [
    cell.get("execution_count")
    for cell in code_cells
]

error_cells = [
    index
    for index, cell in enumerate(code_cells, start=1)
    if any(
        output.get("output_type") == "error"
        for output in cell.get("outputs", [])
    )
]

number_pattern = re.compile(r"^\s*#\s*Code Cell\s+(\d+)\s*$")
code_numbers = []

for cell in code_cells:
    source = "".join(cell.get("source", []))
    first_line = source.splitlines()[0] if source.splitlines() else ""
    match = number_pattern.match(first_line)

    code_numbers.append(
        int(match.group(1))
        if match
        else None
    )

expected_numbers = list(range(1, len(code_cells) + 1))

unexecuted_cells = [
    index
    for index, count in enumerate(execution_counts, start=1)
    if count is None
]

final_marker_found = any(
    "NOTEBOOK 03 FINAL ACCEPTANCE: PASS"
    in "".join(
        output.get("text", [])
        if isinstance(output.get("text"), list)
        else str(output.get("text", ""))
    )
    for cell in code_cells
    for output in cell.get("outputs", [])
    if output.get("output_type") == "stream"
)

health_pass = (
    len(code_cells) == 20
    and not unexecuted_cells
    and not error_cells
    and code_numbers == expected_numbers
    and final_marker_found
)

print("NOTEBOOK 03 HEALTH CHECK")
print("-" * 80)
print("Notebook:", notebook_path.relative_to(project_root))
print("Total cells:", len(cells))
print("Code cells:", len(code_cells))
print("Executed code cells:", len(code_cells) - len(unexecuted_cells))
print("Unexecuted code cells:", len(unexecuted_cells))
print("Error cells:", len(error_cells))
print("Numbered code cells:", sum(number is not None for number in code_numbers))
print(
    "Sequential numbering:",
    "PASS" if code_numbers == expected_numbers else "FAIL",
)
print(
    "Final acceptance marker:",
    "PASS" if final_marker_found else "FAIL",
)

if unexecuted_cells:
    print("Unexecuted cell positions:", unexecuted_cells)

if error_cells:
    print("Error cell positions:", error_cells)

if code_numbers != expected_numbers:
    print("Observed numbering:", code_numbers)
    print("Expected numbering:", expected_numbers)

print()
print(
    "NOTEBOOK 03 HEALTH:",
    "PASS" if health_pass else "FAIL",
)
print(
    "RESTART KERNEL → RUN ALL GATE:",
    "PASS" if health_pass else "FAIL",
)

NOTEBOOK 03 HEALTH CHECK
--------------------------------------------------------------------------------
Notebook: notebooks/phase2/03_book_inventory.ipynb
Total cells: 40
Code cells: 20
Executed code cells: 20
Unexecuted code cells: 0
Error cells: 0
Numbered code cells: 20
Sequential numbering: PASS
Final acceptance marker: PASS

NOTEBOOK 03 HEALTH: PASS
RESTART KERNEL → RUN ALL GATE: PASS
